# v8 · retrieve → verify → fuse · lexical + Hyena retrieval, listwise Llama verifier, stacked fusion

```
judgment ─► sentences ─► InLegalBERT (frozen, overlapping windows) ─► sentence vectors
   ├─► Hyena encoder ×5 (cross-fitted; asymmetric loss; label correlation) ─► p_enc, sentence importance, evidence
   └─► TF-IDF word 1–2 grams + one-vs-rest logistic regression ───────────────► p_lex
shortlist = top-12 of  w·p_enc + (1−w)·p_lex            (w chosen on out-of-fold TRAIN scores)
Llama-3.2-3B QLoRA verifier, LISTWISE: one prompt per judgment (opening + key passages + all 12 candidates with title
   and evidence) → "Section 302: Yes / Section 34: No / …", P(yes) read from the Yes/No logits, 2 candidate orders
stacked fusion: logistic regression on [p_enc, p_lex, blend, p_llm, rank, prior] fitted on DEV → sections
```
**What's new vs v7** (same data, same split, same 100 labels, same input views)
* **Second retriever.** A lexical model reads the words of the judgment (“dowry”, “stab”, “cheque”) and is blended with the
  Hyena encoder. Goal: raise shortlist recall above v7's 78%, the main cap on v7's recall.
* **Larger shortlist (12)** and a **listwise verifier**: the whole shortlist is judged in one prompt, so 12 candidates cost
  little more than a few; candidates carry the section title and an evidence sentence from label-wise attention.
* **Stronger encoder training**: 5 cross-fitted models (v7: 3), asymmetric loss for rare sections, label-correlation layer,
  sentence dropout, 8 epochs on cached sentence vectors (InLegalBERT runs once, not every epoch).
* **Learned fusion** instead of a fixed α: a small logistic-regression stacker fitted on the DEV split; its threshold is
  chosen by 5-fold cross-validation on dev. Test labels are used only for the final scores.
* **Evaluation**: all 463 test judgments (v7: 150) **plus** the first 150 (v7 protocol), both views, 95% bootstrap CIs and
  paired CIs against the encoder alone and against v7-style fixed-weight fusion.
* **Oracle diagnostics** (perfect verifier on the shortlist, gold-injected shortlist, threshold chosen on test) show where
  the remaining headroom is. They use test labels, are written to a separate `DIAGNOSTICS_ORACLE_not_results.csv`,
  and must not be reported as results.

**Running it:** Accelerator **GPU T4 x2** · Internet **On** · Secret **`HF_TOKEN`** (a **Write** token) → **Run All**
(or **Save Version → Save & Run All**). Outputs go to `<you>/hyena-llama-legal-v8`; earlier repos are not touched.
Every stage saves its output and is skipped on a restart.

**Time (T4 x2, estimate):** data restore ~2 min · E sentence vectors ~25 min · L lexical ~10 min · H encoders ~20 min ·
V verifier training ~1.5–2 h · I verifier on dev + test ~1 h · F/T ~5 min (≈ 3.5–4.5 h).

## 1 · Install

In [ ]:
!pip install -q -U "transformers>=4.56,<5" peft bitsandbytes accelerate scikit-learn
!pip install -q pymupdf pypdf
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

## 2 · Hugging Face login (checkpoint sync)

In [ ]:
import os
from huggingface_hub import whoami
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    info = whoami(token=os.environ["HF_TOKEN"])
    role = info.get("auth", {}).get("accessToken", {}).get("role", "?")
    print(f"logged in as {info['name']} · token type: {role}")
    if role == "read":
        print("⚠ READ token: outputs cannot be pushed. Create a WRITE token for sync + resume.")
    print("\nOn your laptop, run:\n"
          f"  python sync_checkpoints.py --repo {info['name']}/hyena-llama-legal-v8 --token <your token>")
except Exception as e:
    print(f"No usable HF_TOKEN secret ({type(e).__name__}). Training still works, but outputs stay in /kaggle/working only.")

## 3 · Settings (edit here)

In [ ]:
%%writefile /kaggle/working/hybrid_config.py
# ═════════════ hybrid_config.py · v8 · lexical + Hyena retrieval → listwise Llama verifier → stacked fusion · edit settings here ═════════════
BASE_MODEL    = "unsloth/Llama-3.2-3B-Instruct"  # ungated mirror of meta-llama/Llama-3.2-3B-Instruct (Llama 3.2 licence applies)
LEGAL_ENCODER = "law-ai/InLegalBERT"             # pretrained on Indian legal text (MIT licence), frozen
import os
WORK_DIR     = os.environ.get("HYBRID_WORK_DIR", "/kaggle/working")
DATA_DIR     = f"{WORK_DIR}/data_v4"              # prepared datasets (same data as v6/v7, written by prepare_data.py)
LEGACY_DATA_DIR = f"{WORK_DIR}/data"              # v3 data, upgraded to v4 if present
CKPT_DIR     = f"{WORK_DIR}/checkpoints_v8"       # stage outputs + checkpoints (also pushed to the Hub)
CACHE_DIR    = f"{WORK_DIR}/cache_v8"             # InLegalBERT sentence vectors (computed once, reused by every stage)
OUT_DIR      = f"{WORK_DIR}/hybrid_legal_v8"      # final model + results
SEED         = 42

# ── checkpoint sync: Kaggle → private Hugging Face repo → your laptop (run sync_checkpoints.py locally) ──
PUSH_TO_HUB  = True                               # needs a WRITE token in the Kaggle secret HF_TOKEN
HF_REPO      = ""                                 # "" = <your-hf-username>/hyena-llama-legal-v8 (private)
HF_DATA_REPO = ""                                 # "" = <your-hf-username>/hyena-llama-legal (prepared data, shared with v4–v7)
CKPT_EVERY_UPDATES = 25                           # verifier checkpoint every N optimizer updates

# ── data (unchanged from v6/v7: same corpus, same temporal split, same 100 labels) ──
DATA_SOURCE  = "aws"                              # "aws" = Indian Supreme Court judgments (S3) | "iltur" = IL-TUR LSI
AWS_YEARS    = list(range(2005, 2025))
AWS_CACHE    = f"{WORK_DIR}/sc_corpus.parquet"
TRAIN_UNTIL, DEV_UNTIL = 2020, 2022               # temporal split: train ≤2020 · dev 2021–22 · test ≥2023
N_LABEL_CLASSES, MIN_LABEL_FREQ = 100, 15
FACTS_PROB   = 0.5                                # training: share of documents shown WITHOUT the sentences that cite sections
MAX_DOC_TOKENS = 16384                            # judgment length read by every stage (Llama tokens; longer = cut)

# ── stage L: lexical retriever (TF-IDF word 1–2 grams + one-vs-rest logistic regression, CPU) ──
LEX_MAX_FEATURES, LEX_C_GRID = 200_000, [2.0, 4.0, 8.0]

# ── stage H: Hyena encoder, cross-fitted ensemble (trained on cached sentence vectors) ──
ENC_DIM, ENC_LAYERS, N_MODES, BAND_CUTOFF = 512, 4, 8, 0.25
ENC_FOLDS, ENC_EPOCHS, ENC_LR = 5, 8, 3e-4        # 5 folds → out-of-fold scores on train + 5-model ensemble on dev/test
ENC_SENT_DROP = 0.10                              # augmentation: drop 10% of sentence vectors in training
ASL_GAMMA_NEG, ASL_CLIP = 4.0, 0.05               # asymmetric loss for the long tail of rare sections
ENTROPY_FLOOR = 0.3

# ── stage S: shortlist = top-K of the lexical/encoder blend ──
SHORTLIST_K  = 12                                 # v7 used a smaller shortlist (recall 78%); the verifier reads all K
TRAIN_INJECT_GOLD = 0.5                           # TRAINING ONLY: prob. of adding gold sections the shortlist missed

# ── stage V: listwise Llama verifier (QLoRA) ──
HEAD_TOKENS, PASSAGE_TOKENS = 384, 1280           # opening of the judgment + top-importance passages (Llama tokens)
EVIDENCE_CHARS = 300                              # one evidence sentence per candidate (label-wise attention)
VERIFIER_EPOCHS = 1
VERIFY_ORDERS = 2                                 # inference: candidates in score order and reversed, averaged
LORA_R, LORA_ALPHA, VERIFIER_LR = 16, 32, 2e-4
EFFECTIVE_BATCH = 8                               # examples per optimizer update (split across GPUs)

# ── stage F/T: fusion + evaluation ──
N_EVAL = 10_000                                   # test judgments for the full pipeline (all 463 by default; v7 used 150)
V7_SUBSET = 150                                   # also reported: the first 150 test judgments (v7 protocol)
BOOTSTRAP = 1000                                  # paired bootstrap resamples for 95% confidence intervals
RUN_ORACLE_DIAGNOSTICS = True                     # ceiling analyses (use test labels, saved separately, never reported as results)

SYSTEM = ("You are a legal assistant for Indian criminal law. You read passages from a court judgment in which the "
          "section numbers have been hidden, and you decide for each candidate Indian Penal Code (IPC) section whether "
          "it applies to the case. Answer every candidate with Yes or No.")

## 4 · Code files (no need to edit)

In [ ]:
%%writefile /kaggle/working/legal_hybrid.py
"""
legal_hybrid.py (v8): retrieve → verify → fuse.

  judgment ─► sentences ─► InLegalBERT (frozen, overlapping windows) ─► sentence vectors
     ├─► Hyena encoder ensemble (5 cross-fitted models, bidirectional multi-band Hyena, label-wise attention,
     │      label-correlation layer, asymmetric loss)  → p_enc (100), sentence importance, evidence per section
     └─► lexical retriever (TF-IDF word 1–2 grams + one-vs-rest logistic regression)  → p_lex (100)
  shortlist = top-K of the blend  w·p_enc + (1−w)·p_lex           (K = 12; w chosen on out-of-fold TRAIN scores)
  Llama-3.2-3B QLoRA verifier, LISTWISE: one prompt per judgment (opening + key passages + every candidate with its
     title and evidence sentence) → "Section 302: Yes / Section 34: No / …";  P(yes) read from the Yes/No logits
  stacked fusion: logistic regression over [p_enc, p_lex, p_llm, blend, rank, label prior] fitted on DEV only,
     threshold chosen on DEV by cross-validation → sections.
"""
import math, re, random, json
from dataclasses import dataclass
from typing import List, Optional, Dict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# ═══════════════════════════════ Hyena encoder (from v6/v7) ═══════════════════════════════
def next_pow2(n: int) -> int:
    return 1 << (int(n) - 1).bit_length()


class MultiBandFilter(nn.Module):
    """Long-convolution filter with LOW (damped oscillatory modes, document-wide memory) and HIGH (short explicit taps,
    local detail) bands combined in the frequency domain with a learnable soft cutoff per channel. Bidirectional."""

    def __init__(self, d: int, n_modes: int = 8, short_len: int = 7, cutoff: float = 1 / 16,
                 tau: float = 0.01, bidirectional: bool = True):
        super().__init__()
        self.d, self.tau, self.bidirectional = d, tau, bidirectional
        nd = 2 if bidirectional else 1
        self.log_rate = nn.Parameter(torch.empty(nd, d, n_modes).uniform_(math.log(1e-4), math.log(5e-2)))
        self.freq = nn.Parameter(torch.rand(nd, d, n_modes) * math.pi * cutoff)
        self.residues = nn.Parameter(torch.randn(nd, d, n_modes) / math.sqrt(n_modes))
        self.short = nn.Parameter(torch.randn(nd, d, short_len) / short_len)
        c = min(max(cutoff, 1e-3), 0.999)
        self.cutoff_logit = nn.Parameter(torch.full((d,), math.log(c / (1 - c))))

    def long_kernel(self, L: int) -> torch.Tensor:
        t = torch.arange(L, device=self.residues.device, dtype=torch.float32)
        rate = self.log_rate.float().exp()
        h = torch.zeros(*self.residues.shape[:2], L, device=t.device)
        for m in range(rate.shape[-1]):
            a = rate[..., m:m + 1]
            h = h + self.residues[..., m:m + 1].float() * a * torch.exp(-a * t) * torch.cos(self.freq[..., m:m + 1].float() * t)
        return h

    def band_masks(self, n_fft: int, device):
        omega = torch.linspace(0, 1, n_fft // 2 + 1, device=device)
        cutoff = torch.sigmoid(self.cutoff_logit.float())[:, None]
        m_low = torch.sigmoid((cutoff - omega) / self.tau)
        return m_low, 1 - m_low

    def forward(self, u: torch.Tensor):
        L = u.shape[-1]
        n = next_pow2(2 * L)
        m_low, m_high = self.band_masks(n, u.device)
        H_low = torch.fft.rfft(self.long_kernel(L), n=n) * m_low
        H_high = torch.fft.rfft(self.short.float(), n=n) * m_high
        y_low = y_high = 0
        for i in range(H_low.shape[0]):
            x = u.float() if i == 0 else u.float().flip(-1)
            X = torch.fft.rfft(x, n=n)
            yl = torch.fft.irfft(X * H_low[i], n=n)[..., :L]
            yh = torch.fft.irfft(X * H_high[i], n=n)[..., :L]
            if i == 1:
                yl, yh = yl.flip(-1), yh.flip(-1)
            y_low, y_high = y_low + yl, y_high + yh
        return y_low, y_high


class HyenaBlock(nn.Module):
    """Hyena operator y = W_o·(q ⊙ (h ∗ (k ⊙ v))) with the filter split into low/high bands."""

    def __init__(self, d: int, dropout: float = 0.1, **filter_kw):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.in_proj = nn.Linear(d, 3 * d)
        self.short_conv = nn.Conv1d(3 * d, 3 * d, 3, padding=1, groups=3 * d)
        self.filter = MultiBandFilter(d, **filter_kw)
        self.out_proj = nn.Linear(d, d, bias=False)
        self.out_bias = nn.Parameter(torch.zeros(d))
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor):
        z = self.short_conv(self.in_proj(self.norm1(x)).transpose(1, 2))
        q, k, v = z.chunk(3, dim=1)
        y_low, y_high = self.filter(k * v)
        low = self.out_proj((q.float() * y_low).transpose(1, 2).to(x.dtype))
        high = self.out_proj((q.float() * y_high).transpose(1, 2).to(x.dtype))
        x = x + self.drop(low + high + self.out_bias)
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x, low, high


class SentenceEmbedder:
    """Frozen InLegalBERT → one vector per sentence. The whole judgment is tokenised as one stream and read in overlapping
    510-token windows (stride 255); every token takes its state from the window where it sits closest to the centre.
    A sentence vector is the mean of its tokens. No sentence is truncated."""

    def __init__(self, model, tokenizer, device, dtype=torch.float32, window: int = 510, batch: int = 16, **_):
        self.model, self.tok, self.device = model.to(device).eval(), tokenizer, torch.device(device)
        self.dtype, self.window, self.batch = dtype, window, batch
        for p in self.model.parameters():
            p.requires_grad_(False)
        self.dim = self.model.config.hidden_size

    @classmethod
    def load(cls, name: str, device, dtype=None, **kw):
        from transformers import AutoTokenizer, AutoModel
        dev = torch.device(device)
        dtype = dtype or (torch.float16 if dev.type == "cuda" else torch.float32)
        return cls(AutoModel.from_pretrained(name, dtype=dtype), AutoTokenizer.from_pretrained(name), dev, dtype, **kw)

    @torch.no_grad()
    def __call__(self, sentences: List[str]) -> torch.Tensor:
        dev = self.device
        ids = self.tok(list(sentences), add_special_tokens=False)["input_ids"]
        flat, sid = [], []
        for j, t in enumerate(ids):
            t = t or [self.tok.unk_token_id]
            flat += t; sid += [j] * len(t)
        N, W = len(flat), self.window
        starts = [0] if N <= W else sorted(set(list(range(0, N - W + 1, W // 2)) + [N - W]))
        states = torch.zeros(N, self.dim, device=dev)
        best = torch.full((N,), float("inf"), device=dev)
        cls_id, sep_id, pad_id = self.tok.cls_token_id, self.tok.sep_token_id, self.tok.pad_token_id or 0
        for b in range(0, len(starts), self.batch):
            group = starts[b:b + self.batch]
            seqs = [[cls_id] + flat[s:s + W] + [sep_id] for s in group]
            T = max(map(len, seqs))
            x = torch.tensor([q + [pad_id] * (T - len(q)) for q in seqs], device=dev)
            m = torch.tensor([[1] * len(q) + [0] * (T - len(q)) for q in seqs], device=dev)
            with torch.autocast(dev.type, dtype=self.dtype, enabled=dev.type == "cuda"):
                h = self.model(input_ids=x, attention_mask=m).last_hidden_state.float()
            for r, s in enumerate(group):
                n = min(W, N - s)
                pos = torch.arange(s, s + n, device=dev)
                dist = (pos - (s + n / 2)).abs()
                upd = dist < best[pos]
                states[pos[upd]] = h[r, 1:1 + n][upd]
                best[pos[upd]] = dist[upd]
        sid = torch.tensor(sid, device=dev)
        S = len(sentences)
        out = torch.zeros(S, self.dim, device=dev).index_add_(0, sid, states)
        cnt = torch.zeros(S, device=dev).index_add_(0, sid, torch.ones(N, device=dev))
        return out / cnt.clamp_min(1)[:, None]


class LegalHyenaEncoder(nn.Module):
    """Hyena encoder over the sequence of sentence vectors (the whole document). Outputs:
       scores (S,) sentence importance · logits (n_labels,) IPC logits from label-wise attention (+ label correlation)
       attn (n_labels, S) which sentences each section looked at (evidence)."""

    def __init__(self, in_dim: int = 768, d: int = 512, n_layers: int = 4, n_labels: int = 100, band_cutoff: float = 0.25,
                 n_modes: int = 8, short_len: int = 3, dropout: float = 0.1, label_init: Optional[torch.Tensor] = None):
        super().__init__()
        self.register_buffer("emb_mean", torch.zeros(in_dim))
        self.register_buffer("emb_std", torch.ones(in_dim))
        self.in_norm = nn.LayerNorm(in_dim)
        self.inp = nn.Linear(in_dim, d)
        self.blocks = nn.ModuleList([HyenaBlock(d, dropout=dropout, n_modes=n_modes, short_len=short_len,
                                                cutoff=band_cutoff) for _ in range(n_layers)])
        self.norm_low, self.norm_high, self.norm_out = nn.LayerNorm(d), nn.LayerNorm(d), nn.LayerNorm(d)
        self.score_head = nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.Linear(d, 1))
        self.label_query = nn.Parameter(torch.randn(n_labels, d) / math.sqrt(d))
        self.label_from_desc = nn.Linear(in_dim, d, bias=False)
        self.register_buffer("label_desc", label_init.float() if label_init is not None else torch.zeros(n_labels, in_dim))
        self.k, self.v = nn.Linear(d, d, bias=False), nn.Linear(d, d)
        self.label_w = nn.Parameter(torch.randn(n_labels, d) / math.sqrt(d))
        self.label_b = nn.Parameter(torch.full((n_labels,), -2.0))
        self.corr = nn.Linear(n_labels, n_labels, bias=False)          # label correlation (e.g. 302 ↔ 34), starts at zero
        nn.init.zeros_(self.corr.weight)

    def forward(self, sent_emb: torch.Tensor):
        z = (sent_emb.float() - self.emb_mean) / self.emb_std
        x = self.inp(self.in_norm(z))[None]
        low_sum = high_sum = 0
        for blk in self.blocks:
            x, low, high = blk(x)
            low_sum, high_sum = low_sum + low, high_sum + high
        x = self.norm_out(x)
        s_low, s_high = self.norm_low(low_sum)[0], self.norm_high(high_sum)[0]
        scores = self.score_head(torch.cat([s_high, x[0]], -1)).squeeze(-1)
        q = self.label_query + self.label_from_desc((self.label_desc - self.emb_mean) / self.emb_std)
        att = torch.softmax(q @ self.k(x[0]).T / math.sqrt(q.shape[-1]), dim=-1)
        ctx = att @ self.v(x[0] + s_low)
        base = (ctx * self.label_w).sum(-1) + self.label_b
        logits = base + self.corr(torch.sigmoid(base))
        return {"scores": scores, "logits": logits, "attn": att}


def asl_loss(logits: torch.Tensor, y: torch.Tensor, gamma_neg: float = 4.0, clip: float = 0.05) -> torch.Tensor:
    """Asymmetric loss (Ridnik et al., ICCV 2021): easy negatives are down-weighted (γ−) and probability-shifted (clip),
    which suits a 100-label problem where most sections are absent from most judgments."""
    p = torch.sigmoid(logits.float())
    p_neg = (p - clip).clamp(min=0) if clip > 0 else p
    lp = y * torch.log(p.clamp_min(1e-8))
    ln = (1 - y) * torch.log((1 - p_neg).clamp_min(1e-8)) * p_neg.pow(gamma_neg)
    return -(lp + ln).sum() / max(1.0, float(y.sum().item()))


@torch.no_grad()
def ensemble_forward(encoders, sent_emb: torch.Tensor):
    """Mean probabilities, importance scores and label attention over the cross-fitted encoders."""
    ps, ss, aa = [], [], []
    for e in encoders:
        o = e(sent_emb)
        ps.append(torch.sigmoid(o["logits"].float())); ss.append(o["scores"].float()); aa.append(o["attn"].float())
    return torch.stack(ps).mean(0), torch.stack(ss).mean(0), torch.stack(aa).mean(0)


# ═══════════════════════════════ documents and views (from v6/v7) ═══════════════════════════════
@dataclass
class Doc:
    ids: torch.Tensor
    sent_ids: torch.Tensor
    sentences: List[str]
    counts: List[int]
    flags: Optional[List[bool]] = None


def split_sentences(text: str, min_chars: int = 25) -> List[str]:
    parts = re.split(r"(?<=[.?!;:])\s+|\n{2,}", re.sub(r"[ \t]+", " ", text))
    out, buf = [], ""
    for p in (p.strip() for p in parts):
        if not p:
            continue
        buf = f"{buf} {p}".strip() if buf else p
        if len(buf) >= min_chars:
            out.append(buf); buf = ""
    if buf:
        out.append(buf)
    return out


def prepare_doc(sentences: List[str], tokenizer, max_tokens: int, flags: Optional[List[bool]] = None) -> Doc:
    enc = tokenizer([" " + s.strip() for s in sentences], add_special_tokens=False)["input_ids"]
    ids, sid, kept, counts, kf = [], [], [], [], []
    flags = flags if flags is not None else [False] * len(sentences)
    for s, t, fl in zip(sentences, enc, flags):
        if not t:
            continue
        if len(ids) + len(t) > max_tokens:
            t = t[: max_tokens - len(ids)]
            if len(t) < 8:
                break
        sid += [len(kept)] * len(t); ids += t; kept.append(s.strip()); counts.append(len(t)); kf.append(bool(fl))
        if len(ids) >= max_tokens:
            break
    if not kept:
        kept, counts, kf, ids, sid = ["(empty)"], [1], [False], [tokenizer.eos_token_id or 0], [0]
    return Doc(torch.tensor(ids, dtype=torch.long), torch.tensor(sid, dtype=torch.long), kept, counts, kf)


PLACEHOLDER_RE = re.compile(r"\[(?:SECTION|CODE)\]")
_USS_RE = re.compile(r"\b[Uu]/[Ss]{1,2}\.?\s*\d{1,3}[A-Z]{0,2}(?:\s*\(\s*[0-9a-z]{1,3}\s*\))*(?:\s*(?:,|/|&|and|or)\s*\d{1,3}[A-Z]{0,2}(?:\s*\(\s*[0-9a-z]{1,3}\s*\))*)*"
                     r"|\b[Ss]{2}\.\s*\d{1,3}[A-Z]{0,2}(?:\s*(?:,|/|&|and|or)\s*\d{1,3}[A-Z]{0,2})*")


def clean_sentences(sentences: List[str]):
    out, flags = [], []
    for s in sentences:
        flags.append("[SECTION]" in s)
        out.append(re.sub(r"\s{2,}", " ", PLACEHOLDER_RE.sub(" ", s)).strip())
    return out, flags


def view_sentences(sentences: List[str], view: str = "full"):
    """full : citation numbers hidden, all sentences kept · facts: every sentence that cited a section is DROPPED."""
    sentences = [_USS_RE.sub("[SECTION]", x) for x in sentences]
    s, flags = clean_sentences(sentences)
    if view == "facts":
        kept = [(a, f) for a, f in zip(s, flags) if not f]
        if len(kept) >= 3:
            s, flags = [a for a, _ in kept], [False] * len(kept)
    return s, flags


SEC_CITE_RE = None


def hide_cited_sections(text: str) -> str:
    """Hide statute citations in a raw document (same rules as data preparation)."""
    global SEC_CITE_RE
    if SEC_CITE_RE is None:
        num = r"\d{1,3}(?!\d)(?:-?(?-i:[A-Z]{1,2})\b)?(?:\s*\(\s*[0-9a-z]{1,3}\s*\))*(?:\s+Part\s+I{1,2}\b)?"
        join = r"\s*(?:,|/|&|\band\b|\bor\b|r/w|read\s+with|alongwith|along\s+with)\s*(?:(?:[Ss]ections?(?:\(s\))?|[Ss]ecs?\.?|S\.)\s*)?"
        sec = r"(?:\b[Ss]ections?(?:\s*\(s\))?|\b[Ss]ecs?\.?|(?<![A-Za-z.])S\.|\b[Uu]/[Ss]{1,2}\.?|\b[Ss]{2}\.|\b[Uu]nder\s+[Ss]ection)"
        SEC_CITE_RE = (re.compile(rf"{sec}\s*{num}(?:{join}{num})*", re.I),
                       re.compile(r"\b(?:Indian\s+Penal\s+Code|I\.\s?P\.\s?C\.?|IPC)\b", re.I))
    a, b = SEC_CITE_RE
    return b.sub("[CODE]", a.sub("[SECTION]", text))


def sec_key(s: str):
    m = re.findall(r"\d+", s); return (int(m[0]) if m else 0, s)


# Titles for sections the v4 data has no description for (the verifier reads the title of every candidate).
EXTRA_IPC_TITLES = {
    "Section 107": "Abetment of a thing", "Section 120A": "Definition of criminal conspiracy",
    "Section 141": "Unlawful assembly", "Section 143": "Punishment for being member of unlawful assembly",
    "Section 144": "Joining unlawful assembly armed with deadly weapon", "Section 186": "Obstructing public servant in discharge of public functions",
    "Section 193": "Punishment for false evidence", "Section 299": "Culpable homicide", "Section 300": "Murder",
    "Section 308": "Attempt to commit culpable homicide", "Section 309": "Attempt to commit suicide",
    "Section 312": "Causing miscarriage", "Section 313": "Causing miscarriage without woman's consent",
    "Section 319": "Hurt", "Section 320": "Grievous hurt", "Section 332": "Voluntarily causing hurt to deter public servant from his duty",
    "Section 336": "Act endangering life or personal safety of others",
    "Section 337": "Causing hurt by act endangering life or personal safety of others",
    "Section 338": "Causing grievous hurt by act endangering life or personal safety of others",
    "Section 343": "Wrongful confinement for three or more days", "Section 344": "Wrongful confinement for ten or more days",
    "Section 353": "Assault or criminal force to deter public servant from discharge of his duty",
    "Section 354A": "Sexual harassment", "Section 354B": "Assault or use of criminal force to woman with intent to disrobe",
    "Section 354D": "Stalking", "Section 359": "Kidnapping", "Section 361": "Kidnapping from lawful guardianship",
    "Section 362": "Abduction", "Section 364A": "Kidnapping for ransom",
    "Section 365": "Kidnapping or abducting with intent secretly and wrongfully to confine person",
    "Section 366A": "Procuration of minor girl", "Section 368": "Wrongfully concealing or keeping in confinement kidnapped or abducted person",
    "Section 370": "Trafficking of person", "Section 375": "Rape", "Section 376D": "Gang rape", "Section 378": "Theft",
    "Section 381": "Theft by clerk or servant of property in possession of master", "Section 383": "Extortion",
    "Section 386": "Extortion by putting a person in fear of death or grievous hurt", "Section 390": "Robbery",
    "Section 391": "Dacoity", "Section 393": "Attempt to commit robbery",
    "Section 398": "Attempt to commit robbery or dacoity when armed with deadly weapon",
    "Section 399": "Making preparation to commit dacoity", "Section 402": "Assembling for purpose of committing dacoity",
    "Section 403": "Dishonest misappropriation of property", "Section 405": "Criminal breach of trust",
    "Section 408": "Criminal breach of trust by clerk or servant",
    "Section 412": "Dishonestly receiving property stolen in the commission of a dacoity", "Section 415": "Cheating",
    "Section 417": "Punishment for cheating", "Section 425": "Mischief", "Section 426": "Punishment for mischief",
    "Section 427": "Mischief causing damage to the amount of fifty rupees",
    "Section 435": "Mischief by fire or explosive substance with intent to cause damage", "Section 441": "Criminal trespass",
    "Section 442": "House-trespass", "Section 449": "House-trespass in order to commit offence punishable with death",
    "Section 450": "House-trespass in order to commit offence punishable with imprisonment for life",
    "Section 451": "House-trespass in order to commit offence punishable with imprisonment",
    "Section 457": "Lurking house-trespass or house-breaking by night in order to commit offence punishable with imprisonment",
    "Section 463": "Forgery", "Section 464": "Making a false document",
    "Section 466": "Forgery of record of Court or of public register", "Section 474": "Having possession of document known to be forged and intending to use it as genuine",
    "Section 477A": "Falsification of accounts", "Section 489A": "Counterfeiting currency-notes or bank-notes",
    "Section 489B": "Using as genuine forged or counterfeit currency-notes or bank-notes",
    "Section 494": "Marrying again during lifetime of husband or wife", "Section 499": "Defamation",
    "Section 500": "Punishment for defamation", "Section 503": "Criminal intimidation",
    "Section 505": "Statements conducing to public mischief", "Section 511": "Punishment for attempting to commit offences punishable with imprisonment for life or other imprisonment",
}


def section_title(sec: str, statute_desc: Dict[str, str], n: int = 110) -> str:
    d = (statute_desc.get(sec) or EXTRA_IPC_TITLES.get(sec, "")).strip()
    return (d[:n].rsplit(" ", 1)[0] + "…") if len(d) > n else d


# ═══════════════════════════════ metrics ═══════════════════════════════
def micro_prf(golds, preds):
    tp = sum(len(g & p) for g, p in zip(golds, preds))
    fp = sum(len(p - g) for g, p in zip(golds, preds))
    fn = sum(len(g - p) for g, p in zip(golds, preds))
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    return {"precision": round(P, 3), "recall": round(R, 3), "f1": round(2 * P * R / (P + R) if P + R else 0.0, 3)}


def micro_f1_raw(golds, preds) -> float:
    tp = sum(len(g & p) for g, p in zip(golds, preds)); fp = sum(len(p - g) for g, p in zip(golds, preds))
    fn = sum(len(g - p) for g, p in zip(golds, preds))
    return 2 * tp / (2 * tp + fp + fn) if tp else 0.0


def macro_f1(golds, preds, labels) -> float:
    f = []
    for c in labels:
        tp = sum((c in g) and (c in p) for g, p in zip(golds, preds))
        fp = sum((c not in g) and (c in p) for g, p in zip(golds, preds))
        fn = sum((c in g) and (c not in p) for g, p in zip(golds, preds))
        if tp + fp + fn:
            f.append(2 * tp / (2 * tp + fp + fn))
    return round(float(np.mean(f)), 3) if f else 0.0


def doc_f1(g, p) -> float:
    g, p = set(g), set(p)
    return 2 * len(g & p) / (len(g) + len(p)) if (g or p) else 1.0


def ranking_metrics(golds, ranked):
    vr = [(g, r) for g, r in zip(golds, ranked) if g]
    out = {f"recall@{k}": round(float(np.mean([len(set(r[:k]) & g) / len(g) for g, r in vr])), 3) for k in (1, 5, 10)}
    out["precision@1"] = round(float(np.mean([bool(r) and r[0] in g for g, r in vr])), 3)
    return out


def bootstrap_ci(golds, preds_a, preds_b=None, n: int = 1000, seed: int = 0):
    """95% CI of micro-F1 (or of the paired difference a − b) by resampling judgments."""
    rng = np.random.default_rng(seed); N = len(golds); vals = []
    for _ in range(n):
        ix = rng.integers(0, N, N)
        g = [golds[i] for i in ix]
        v = micro_f1_raw(g, [preds_a[i] for i in ix])
        if preds_b is not None:
            v -= micro_f1_raw(g, [preds_b[i] for i in ix])
        vals.append(v)
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return round(float(lo), 3), round(float(hi), 3)


def best_threshold(golds, prob_rows, labels, grid=None):
    """prob_rows: list of {section: prob}. Threshold maximising micro-F1 (used on DEV only)."""
    grid = grid if grid is not None else np.round(np.arange(0.05, 0.96, 0.01), 2)
    best = max(grid, key=lambda th: micro_f1_raw(golds, [{c for c, p in r.items() if p >= th} for r in prob_rows]))
    return float(best)


def decide(prob_row: Dict[str, float], th: float, at_least_one: bool = False) -> set:
    s = {c for c, p in prob_row.items() if p >= th}
    if not s and at_least_one and prob_row:
        s = {max(prob_row, key=prob_row.get)}
    return s


# ═══════════════════════════════ shortlist, passages, evidence ═══════════════════════════════
def shortlist(p_blend: np.ndarray, k: int) -> List[int]:
    return [int(j) for j in np.argsort(-p_blend)[:k]]


def key_passages(doc: Doc, scores: np.ndarray, head_tokens: int, passage_tokens: int, max_chars: int = 600) -> str:
    """Opening of the judgment (≤ head_tokens) + the highest-importance later sentences (≤ passage_tokens), in order."""
    head, used = [], 0
    for j, c in enumerate(doc.counts):
        if used + c > head_tokens:
            break
        head.append(j); used += c
    rest, used = [], 0
    for j in np.argsort(-np.asarray(scores)).tolist():
        if j < len(head) or j >= len(doc.counts):
            continue
        c = min(doc.counts[j], max_chars // 4)
        if used + c > passage_tokens:
            continue
        rest.append(j); used += c
    cut = lambda s: s if len(s) <= max_chars else s[:max_chars].rsplit(" ", 1)[0] + " …"
    txt = "Opening of the judgment:\n" + " ".join(cut(doc.sentences[j]) for j in head) if head else ""
    if rest:
        txt += ("\n\n" if txt else "") + "Key passages:\n" + "\n".join(f"- {cut(doc.sentences[j])}" for j in sorted(rest))
    return txt


def evidence_index(attn: np.ndarray, label: int) -> int:
    return int(np.argmax(attn[label]))


def verify_user(passages: str, cands: List[str], titles: Dict[str, str], evidence: Dict[str, str]) -> str:
    lines = []
    for i, c in enumerate(cands, 1):
        t = titles.get(c, "")
        lines.append(f"{i}. {c}" + (f" ({t})" if t else "") + (f"\n   Possible evidence: \"{evidence[c]}\"" if evidence.get(c) else ""))
    return (f"{passages}\n\nCandidate IPC sections:\n" + "\n".join(lines) +
            "\n\nFor each candidate, in the order given, answer on its own line as '<section>: Yes' if it applies to "
            "this case or '<section>: No' if it does not.")


# ═══════════════════════════════ listwise verifier: training example + scoring ═══════════════════════════════
class VerifierTokens:
    """Token pieces shared by training and inference, so both see exactly the same token sequence."""

    def __init__(self, tok):
        self.tok = tok
        self.yes = tok(" Yes", add_special_tokens=False)["input_ids"]
        self.no = tok(" No", add_special_tokens=False)["input_ids"]
        assert len(self.yes) == 1 and len(self.no) == 1, f"' Yes'/' No' must be single tokens, got {self.yes} {self.no}"
        self.yes, self.no = self.yes[0], self.no[0]
        self.nl = tok("\n", add_special_tokens=False)["input_ids"]
        eot = tok.convert_tokens_to_ids("<|eot_id|>") if "<|eot_id|>" in tok.get_vocab() else tok.eos_token_id
        self.end = [eot]
        self._line = {}

    def line(self, sec: str) -> List[int]:
        if sec not in self._line:
            self._line[sec] = self.tok(f"{sec}:", add_special_tokens=False)["input_ids"]
        return self._line[sec]

    def prompt(self, system: str, user: str) -> List[int]:
        msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        txt = self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        return self.tok(txt, add_special_tokens=False)["input_ids"]


def verifier_example(vt: VerifierTokens, system: str, user: str, cands: List[str], answers: List[bool]):
    """ids (L,), positions of the Yes/No slots, targets (1 = Yes). The model is trained on the slots only."""
    ids = vt.prompt(system, user); pos, tgt = [], []
    for c, a in zip(cands, answers):
        ids += vt.line(c); pos.append(len(ids) - 1)          # logits at this position predict the Yes/No token
        ids += [vt.yes if a else vt.no]; tgt.append(1 if a else 0)
        ids += vt.nl
    ids += vt.end
    return torch.tensor(ids), torch.tensor(pos), torch.tensor(tgt)


def _base(llm):
    return llm.get_base_model() if hasattr(llm, "get_base_model") else llm


def verifier_loss(llm, vt: VerifierTokens, ids, pos, tgt, pos_weight: float = 1.0):
    """2-way cross-entropy over {No, Yes} at every slot (LM head applied at the slots only → small memory)."""
    dev = llm.get_input_embeddings().weight.device
    h = _base(llm).model(input_ids=ids[None].to(dev), use_cache=False).last_hidden_state[0, pos.to(dev)]
    head = _base(llm).lm_head
    W = head.weight[[vt.no, vt.yes]]
    logits = (h.to(W.dtype) @ W.T).float()
    w = torch.tensor([1.0, pos_weight], device=dev)
    return F.cross_entropy(logits, tgt.to(dev), weight=w)


@torch.no_grad()
def score_candidates(llm, vt: VerifierTokens, system: str, user: str, cands: List[str]) -> List[float]:
    """One prefill of the prompt, then each candidate line with the KV cache: P(Yes) from the Yes/No logits.
    The model's own answer is appended before the next line, exactly as in training (teacher forcing ↔ own answers)."""
    from transformers import DynamicCache
    dev = llm.get_input_embeddings().weight.device
    body, head = _base(llm).model, _base(llm).lm_head
    W = head.weight[[vt.no, vt.yes]]
    cache = DynamicCache()
    pre = vt.prompt(system, user)
    body(input_ids=torch.tensor([pre], device=dev), past_key_values=cache, use_cache=True)
    out, n = [], len(pre)
    for c in cands:
        piece = vt.line(c)
        L = len(piece)
        h = body(input_ids=torch.tensor([piece], device=dev), past_key_values=cache, use_cache=True,
                 position_ids=torch.arange(n, n + L, device=dev)[None]).last_hidden_state[0, -1]
        n += L
        q = torch.softmax((h.to(W.dtype) @ W.T).float(), -1)[1].item()
        out.append(q)
        nxt = [vt.yes if q >= 0.5 else vt.no] + vt.nl
        body(input_ids=torch.tensor([nxt], device=dev), past_key_values=cache, use_cache=True,
             position_ids=torch.arange(n, n + len(nxt), device=dev)[None])
        n += len(nxt)
    return out


# ═══════════════════════════════ stacked fusion ═══════════════════════════════
def _logit(p):
    p = np.clip(np.asarray(p, dtype=np.float64), 1e-4, 1 - 1e-4)
    return np.log(p / (1 - p))


def stack_features(p_enc, p_lex, p_blend, p_llm, rank, prior):
    """One row per (judgment, shortlisted section)."""
    return np.stack([_logit(p_enc), _logit(p_lex), _logit(p_blend), _logit(p_llm), np.asarray(rank, float),
                     np.log(np.asarray(prior, float) + 1e-4), _logit(p_llm) * _logit(p_enc)], -1)


# ═══════════════════════════════ loading a trained v8 model (laptop / server) ═══════════════════════════════
def load_trained(out_dir: str, device="cuda:0", base_model: Optional[str] = None, load_4bit: bool = True,
                 legal_encoder: Optional[str] = None):
    """Returns (tokenizer, llm, encoders, ck). ck holds the lexical model, stacker, thresholds and the embedder."""
    import pickle
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import PeftModel
    ck = torch.load(f"{out_dir}/hybrid_model_v8.pt", map_location="cpu", weights_only=False)
    dev = torch.device(device)
    ampere = dev.type == "cuda" and torch.cuda.get_device_capability(dev)[0] >= 8
    dtype = (torch.bfloat16 if ampere else torch.float16) if dev.type == "cuda" else torch.float32
    tok = AutoTokenizer.from_pretrained(f"{out_dir}/llama_lora")
    kw = dict(dtype=dtype)
    if load_4bit and dev.type == "cuda":
        kw.update(device_map={"": dev.index or 0}, quantization_config=BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype))
    base = AutoModelForCausalLM.from_pretrained(base_model or ck["base_model"], **kw)
    if not (load_4bit and dev.type == "cuda"):
        base = base.to(dev)
    llm = PeftModel.from_pretrained(base, f"{out_dir}/llama_lora").eval()
    encoders = []
    for sd in ck["encoder_states"]:
        e = LegalHyenaEncoder(**ck["encoder_config"]).to(dev); e.load_state_dict(sd); e.eval(); encoders.append(e)
    with open(f"{out_dir}/lexical_and_stacker.pkl", "rb") as f:
        ck.update(pickle.load(f))
    ck["dtype"] = dtype
    ck["embedder"] = SentenceEmbedder.load(legal_encoder or ck["legal_encoder"], dev)
    return tok, llm, encoders, ck


class Predictor:
    """End-to-end analysis of one judgment with a trained v8 model."""

    def __init__(self, tok, llm, encoders, ck):
        self.tok, self.llm, self.encoders, self.ck = tok, llm, encoders, ck
        self.vt = VerifierTokens(tok)
        self.dev = next(encoders[0].parameters()).device
        self.labels = ck["label_names"]

    @torch.no_grad()
    def __call__(self, text: str, hide_cited: bool = True, view: str = "full"):
        ck, L = self.ck, self.labels
        if hide_cited:
            text = hide_cited_sections(text)
        sents, flags = view_sentences(split_sentences(text), view)
        doc = prepare_doc(sents, self.tok, ck["max_doc_tokens"], flags)
        with torch.autocast(self.dev.type, dtype=ck["dtype"], enabled=self.dev.type == "cuda"):
            p, s, a = ensemble_forward(self.encoders, ck["embedder"](doc.sentences).to(self.dev))
        p_enc, s, a = p.cpu().numpy(), s.cpu().numpy(), a.cpu().numpy()
        p_lex = ck["lexical"].predict_proba([" ".join(doc.sentences)])[0]
        blend = ck["blend_w"] * p_enc + (1 - ck["blend_w"]) * p_lex
        cand_idx = shortlist(blend, ck["shortlist_k"])
        cands = [L[j] for j in cand_idx]
        titles = {c: section_title(c, ck["statute_desc"]) for c in cands}
        ev = {L[j]: doc.sentences[evidence_index(a, j)][:ck["evidence_chars"]] for j in cand_idx}
        passages = key_passages(doc, s, ck["head_tokens"], ck["passage_tokens"])
        q = {c: 0.0 for c in cands}
        orders = [cands, cands[::-1]][:max(1, ck["verify_orders"])]
        for order in orders:
            for c, v in zip(order, score_candidates(self.llm, self.vt, ck["system_prompt"], verify_user(passages, order, titles, ev), order)):
                q[c] += v / len(orders)
        X = stack_features(p_enc[cand_idx], p_lex[cand_idx], blend[cand_idx], [q[c] for c in cands],
                           np.arange(len(cands)) / ck["shortlist_k"], ck["label_prior"][cand_idx])
        fused = dict(zip(cands, ck["stacker"].predict_proba(X)[:, 1]))
        secs = sorted(decide(fused, ck["threshold"], ck["at_least_one"]), key=sec_key)
        return {"sections": secs, "fused": dict(sorted(fused.items(), key=lambda kv: -kv[1])),
                "verifier": q, "titles": titles, "evidence": ev, "passages": passages,
                "doc_tokens": len(doc.ids), "sentences": len(doc.sentences)}

In [ ]:
%%writefile /kaggle/working/prepare_data.py
"""prepare_data.py (v4, unchanged in v8): build train/dev/test (+ unlabelled) datasets on disk. CPU only; safe to re-run (skips if done).
Order: data_v4 already local → restore data_v4 from the Hub → upgrade v3 data (local or Hub) → build from the AWS corpus.
The v4 upgrade hides citation forms the v3 masking missed ("Section(s) 147, 148", "376(2)(n), 377 & 506") and adds
their sections to the labels."""
import os, re, json, sys, multiprocessing
from collections import Counter
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from hybrid_config import *
from legal_hybrid import split_sentences
from datasets import Dataset, load_dataset
from tqdm.auto import tqdm

META = f"{DATA_DIR}/meta.json"

# ── private Hub repo: restore prepared data in a new session instead of re-downloading the corpus ──
REPO, api = "", None
if PUSH_TO_HUB:
    try:
        from huggingface_hub import HfApi, snapshot_download
        api = HfApi(); REPO = HF_DATA_REPO or f"{api.whoami()['name']}/hyena-llama-legal"
        api.create_repo(REPO, private=True, exist_ok=True)
        if not os.path.exists(META):
            snapshot_download(REPO, allow_patterns=["data_v4/*", "data_v4/**"], local_dir=WORK_DIR)
            if os.path.exists(META):
                print("restored prepared v4 data from the Hub:", REPO)
        if not os.path.exists(META) and not os.path.exists(f"{LEGACY_DATA_DIR}/meta.json"):
            snapshot_download(REPO, allow_patterns=["data/*", "data/**"], local_dir=WORK_DIR)
    except Exception as e:
        api, REPO = None, ""
        print(f"[hub] not available ({type(e).__name__}); data stays local")
if os.path.exists(META):
    print("data already prepared:", DATA_DIR); sys.exit(0)
os.makedirs(DATA_DIR, exist_ok=True)

def sec_key(s):
    m = re.findall(r"\d+", s); return (int(m[0]) if m else 0, s)

IPC_TITLES = {
    "Section 34": "Acts done by several persons in furtherance of common intention", "Section 109": "Punishment of abetment",
    "Section 114": "Abettor present when offence is committed", "Section 120B": "Punishment of criminal conspiracy",
    "Section 147": "Punishment for rioting", "Section 148": "Rioting, armed with deadly weapon",
    "Section 149": "Every member of unlawful assembly guilty of offence committed in prosecution of common object",
    "Section 161": "Public servant taking gratification other than legal remuneration (repealed 1988)",
    "Section 201": "Causing disappearance of evidence of offence", "Section 279": "Rash driving or riding on a public way",
    "Section 302": "Punishment for murder", "Section 304": "Punishment for culpable homicide not amounting to murder",
    "Section 304A": "Causing death by negligence", "Section 304B": "Dowry death", "Section 306": "Abetment of suicide",
    "Section 307": "Attempt to murder", "Section 323": "Punishment for voluntarily causing hurt",
    "Section 324": "Voluntarily causing hurt by dangerous weapons or means", "Section 325": "Punishment for voluntarily causing grievous hurt",
    "Section 326": "Voluntarily causing grievous hurt by dangerous weapons or means", "Section 341": "Punishment for wrongful restraint",
    "Section 342": "Punishment for wrongful confinement", "Section 354": "Assault or criminal force to woman with intent to outrage her modesty",
    "Section 363": "Punishment for kidnapping", "Section 364": "Kidnapping or abducting in order to murder",
    "Section 366": "Kidnapping, abducting or inducing woman to compel her marriage", "Section 376": "Punishment for rape",
    "Section 377": "Unnatural offences", "Section 379": "Punishment for theft", "Section 380": "Theft in dwelling house",
    "Section 384": "Punishment for extortion", "Section 392": "Punishment for robbery",
    "Section 394": "Voluntarily causing hurt in committing robbery", "Section 395": "Punishment for dacoity",
    "Section 396": "Dacoity with murder", "Section 397": "Robbery or dacoity with attempt to cause death or grievous hurt",
    "Section 406": "Punishment for criminal breach of trust", "Section 409": "Criminal breach of trust by public servant, banker, merchant or agent",
    "Section 411": "Dishonestly receiving stolen property", "Section 419": "Punishment for cheating by personation",
    "Section 420": "Cheating and dishonestly inducing delivery of property", "Section 436": "Mischief by fire or explosive substance with intent to destroy house",
    "Section 447": "Punishment for criminal trespass", "Section 448": "Punishment for house-trespass",
    "Section 452": "House-trespass after preparation for hurt, assault or wrongful restraint", "Section 465": "Punishment for forgery",
    "Section 467": "Forgery of valuable security, will, etc.", "Section 468": "Forgery for purpose of cheating",
    "Section 471": "Using as genuine a forged document", "Section 498A": "Husband or relative of husband of a woman subjecting her to cruelty",
    "Section 504": "Intentional insult with intent to provoke breach of the peace", "Section 506": "Punishment for criminal intimidation",
    "Section 509": "Word, gesture or act intended to insult the modesty of a woman",
}

# ── IPC label extraction + leakage masking ──
_NUM = r"\d{1,3}(?!\d)(?:-?(?-i:[A-Z]{1,2})\b)?(?:\s*\(\s*[0-9a-z]{1,3}\s*\))*(?:\s+Part\s+I{1,2}\b)?"
_JOIN = r"\s*(?:,|/|&|\band\b|\bor\b|r/w|read\s+with|alongwith|along\s+with)\s*(?:(?:[Ss]ections?(?:\(s\))?|[Ss]ecs?\.?|S\.)\s*)?"
_LIST = rf"{_NUM}(?:{_JOIN}{_NUM})*"
_SEC = r"(?:\b[Ss]ections?(?:\s*\(s\))?|\b[Ss]ecs?\.?|(?<![A-Za-z.])S\.|\b[Uu]/[Ss]{1,2}\.?|\b[Ss]{2}\.|\b[Uu]nder\s+[Ss]ection)"
IPC_RE = re.compile(rf"{_SEC}\s*({_LIST})\s*,?\s*(?:of\s+(?:the\s+)?)?(?:Indian\s+Penal\s+Code|I\.\s?P\.\s?C\.?|IPC|Penal\s+Code)", re.I)
ANY_SEC_RE = re.compile(rf"{_SEC}\s*{_LIST}", re.I)
IPC_WORD_RE = re.compile(r"\b(?:Indian\s+Penal\s+Code|I\.\s?P\.\s?C\.?|IPC)\b", re.I)
def ipc_sections(text):
    out = set()
    for m in IPC_RE.finditer(text):
        nums = re.sub(r"\(\s*\d+\s*\)", "", m.group(1)).replace("-", "")
        for n in re.findall(r"(?<![\dA-Za-z])\d{1,3}(?-i:[A-Z]{0,2})(?![\d])", nums):
            if 1 <= int(re.match(r"\d+", n).group()) <= 511: out.add(f"Section {n.upper()}")
    return out
def mask_sections(t):
    return IPC_WORD_RE.sub("[CODE]", ANY_SEC_RE.sub("[SECTION]", t))


# ── v4 upgrade: re-mask leftovers and recover their labels ──
_LEFT_RE = re.compile(rf"\[SECTION\]((?:\s*\(\s*[0-9a-z]{{1,3}}\s*\))*(?:{_JOIN}{_NUM})+)", re.I)
_CODE_AFTER = r"\s*,?\s*(?:of\s+(?:the\s+)?)?\[CODE\]"
_LEFT_IPC_RE = re.compile(rf"\[SECTION\]((?:\s*\(\s*[0-9a-z]{{1,3}}\s*\))*(?:{_JOIN}{_NUM})+){_CODE_AFTER}", re.I)
_SEC_IPC_RE = re.compile(rf"{_SEC}\s*({_LIST}){_CODE_AFTER}", re.I)

def _numbers(span):
    span = re.sub(r"\(\s*[0-9a-z]{1,3}\s*\)", "", span).replace("-", "")
    return {f"Section {n.upper()}" for n in re.findall(r"(?<![\dA-Za-z])\d{1,3}(?-i:[A-Z]{0,2})(?![\d])", span)
            if 1 <= int(re.match(r"\d+", n).group()) <= 511}

def upgrade_rows(sentences, label_ids, l2i):
    """Returns (re-masked sentences, labels + recovered labels, n_fixed)."""
    found, out, fixed = set(), [], 0
    for t in sentences:
        for m in _LEFT_IPC_RE.finditer(t):
            found |= _numbers(m.group(1))
        for m in _SEC_IPC_RE.finditer(t):
            found |= _numbers(m.group(1))
        u = _LEFT_RE.sub("[SECTION]", ANY_SEC_RE.sub("[SECTION]", t))
        fixed += u != t; out.append(u)
    return out, sorted(set(label_ids) | {l2i[s] for s in found if s in l2i}), fixed

def upgrade_splits(splits, label_names):
    l2i = {s: i for i, s in enumerate(label_names)}; stats = {}
    for name, ds in list(splits.items()):
        rows, fixed, added = [], 0, 0
        for r in ds:
            t, labs, f = upgrade_rows(r["text"], r["labels"], l2i if name != "unlabelled" else {})
            fixed += f; added += len(labs) - len(r["labels"]); rows.append({"id": r["id"], "text": t, "labels": labs})
        splits[name] = Dataset.from_list(rows); stats[name] = {"sentences_re_masked": fixed, "labels_added": added}
    print("v4 upgrade:", stats)
    return splits, stats

def save(splits, label_names, statute_desc, stats):
    for name, ds in splits.items():
        ds.save_to_disk(f"{DATA_DIR}/{name}")
    json.dump({"label_names": label_names, "statute_desc": statute_desc, "sizes": {k: len(v) for k, v in splits.items()},
               "version": 4, "upgrade": stats}, open(META, "w"), indent=1)
    print("saved:", {k: len(v) for k, v in splits.items()}, "| label classes:", len(label_names))
    if api is not None:
        try:
            api.upload_folder(repo_id=REPO, folder_path=DATA_DIR, path_in_repo="data_v4", commit_message="prepared data v4")
            print("[hub] pushed prepared data → a new session can resume without re-downloading the corpus")
        except Exception as e:
            print(f"[hub] data push failed ({type(e).__name__}: {e})")

if os.path.exists(f"{LEGACY_DATA_DIR}/meta.json"):
    from datasets import load_from_disk
    legacy = json.load(open(f"{LEGACY_DATA_DIR}/meta.json"))
    print("upgrading v3 data from", LEGACY_DATA_DIR)
    splits = {n: load_from_disk(f"{LEGACY_DATA_DIR}/{n}") for n in ["train", "dev", "test", "unlabelled"]}
    splits, stats = upgrade_splits(splits, legacy["label_names"])
    save(splits, legacy["label_names"], legacy["statute_desc"], stats); sys.exit(0)

if DATA_SOURCE == "aws":
    import tarfile, requests, xml.etree.ElementTree as ET, pandas as pd
    import pymupdf as fitz
    from concurrent.futures import ProcessPoolExecutor
    S3_URL = "https://indian-supreme-court-judgments.s3.ap-south-1.amazonaws.com"   # public bucket, plain HTTPS

    def s3_list(prefix):
        keys, token = [], None
        while True:
            params = {"list-type": "2", "prefix": prefix, **({"continuation-token": token} if token else {})}
            r = requests.get(S3_URL + "/", params=params, timeout=60); r.raise_for_status()
            root = ET.fromstring(r.content); ns = {"s": root.tag.split("}")[0].strip("{")}
            keys += [c.findtext("s:Key", namespaces=ns) for c in root.findall("s:Contents", ns)]
            if root.findtext("s:IsTruncated", namespaces=ns) != "true":
                return keys
            token = root.findtext("s:NextContinuationToken", namespaces=ns)

    def s3_download(key, path):
        with requests.get(f"{S3_URL}/{key}", stream=True, timeout=600) as r:
            r.raise_for_status()
            with open(path, "wb") as f:
                for chunk in r.iter_content(1 << 22):
                    f.write(chunk)

    def pdf_text(item):
        name, data = item
        try:
            doc = fitz.open(stream=data, filetype="pdf")
            txt = "\n".join(p.get_text() for p in doc)
        except Exception:
            return name, ""
        txt = re.sub(r"(?im)^.*(signature not verified|digitally signed|reason:|date:\s*\d).*$", " ", txt)
        return name, txt[:200_000]

    def year_docs(year):
        docs = []
        for key in [k for k in s3_list(f"data/tar/year={year}/english/") if k.endswith(".tar")]:
            local = f"/tmp/{key.replace('/', '_')}"
            s3_download(key, local)
            with tarfile.open(local) as tf:
                items = [(os.path.basename(m.name), tf.extractfile(m).read()) for m in tf if m.isfile() and m.name.lower().endswith(".pdf")]
            os.remove(local)
            with ProcessPoolExecutor(os.cpu_count(), mp_context=multiprocessing.get_context("fork")) as ex:
                docs += [{"id": n, "year": year, "text": t} for n, t in ex.map(pdf_text, items, chunksize=8) if len(t) > 3000]
        return docs

    if os.path.exists(AWS_CACHE):
        corpus = pd.read_parquet(AWS_CACHE); print("loaded cache:", AWS_CACHE)
    else:
        rows = []
        for y in tqdm(AWS_YEARS, desc="years"):
            try:
                got = year_docs(y); rows += got; print(f"{y}: {len(got)} judgments with text", flush=True)
            except Exception as e:
                print(f"{y}: skipped ({e})", flush=True)
        corpus = pd.DataFrame(rows); corpus.to_parquet(AWS_CACHE)
    print("judgments:", len(corpus))

    corpus["sections"] = corpus["text"].map(ipc_sections)
    freq = Counter(s for ss in corpus["sections"] for s in ss)
    LABEL_NAMES = sorted([s for s, c in freq.most_common(N_LABEL_CLASSES) if c >= MIN_LABEL_FREQ], key=sec_key)
    L2I = {s: i for i, s in enumerate(LABEL_NAMES)}
    STATUTE_DESC = {s: IPC_TITLES.get(s, "") for s in LABEL_NAMES}

    rows = []
    for r in tqdm(corpus.itertuples(), total=len(corpus), desc="label+mask"):
        txt = re.sub(r"(?<!\n)\n(?!\n)", " ", mask_sections(r.text))
        rows.append({"id": r.id, "year": int(r.year), "text": split_sentences(txt),
                     "labels": sorted(L2I[s] for s in r.sections if s in L2I), "any_ipc": len(r.sections) > 0})
    leaks = sum(bool(ANY_SEC_RE.search(" ".join(r["text"]))) for r in rows[:500])
    print(f"leakage check: {leaks} of first 500 docs still contain a 'Section <n>' mention (should be 0)")
    lab = [r for r in rows if r["labels"]]
    pick = lambda lo, hi: Dataset.from_list([{k: r[k] for k in ("id", "text", "labels")} for r in lab if lo <= r["year"] <= hi])
    splits = {"train": pick(0, TRAIN_UNTIL), "dev": pick(TRAIN_UNTIL + 1, DEV_UNTIL), "test": pick(DEV_UNTIL + 1, 9999),
              "unlabelled": Dataset.from_list([{"id": r["id"], "text": r["text"], "labels": []} for r in rows
                                               if not r["any_ipc"] and r["year"] <= TRAIN_UNTIL] or [{"id": "none", "text": ["."], "labels": []}])}
    print("most common sections:", [f"{s} ({freq[s]})" for s in sorted(LABEL_NAMES, key=lambda s: -freq[s])[:8]])
else:
    lsi = load_dataset("Exploration-Lab/IL-TUR", "lsi")
    LABEL_NAMES = list(lsi["train"].features["labels"].feature.names)
    STATUTE_DESC = {r["id"]: re.sub(r"\s+", " ", " ".join(r["text"])).strip() for r in lsi["statutes"]}
    keep = lambda ds: Dataset.from_list([{"id": str(r["id"]), "text": list(r["text"]), "labels": list(r["labels"])} for r in ds if len(r["labels"]) > 0])
    splits = {"train": keep(lsi["train"]), "dev": keep(lsi["dev"]), "test": keep(lsi["test"]),
              "unlabelled": Dataset.from_list([{"id": "none", "text": ["."], "labels": []}])}

splits = {name: ds.shuffle(seed=SEED) for name, ds in splits.items()}
splits, stats = upgrade_splits(splits, LABEL_NAMES)
save(splits, LABEL_NAMES, STATUTE_DESC, stats)

In [ ]:
%%writefile /kaggle/working/train_v8.py
"""
train_v8.py: multi-GPU training + evaluation of v8 (retrieve → verify → fuse).
Launch (from the notebook, through run_training.py):   torchrun --standalone --nproc_per_node=<GPUs> train_v8.py

Stages (each one saves its output and is skipped on a restart; outputs are pushed to a private Hugging Face repo):
  E  InLegalBERT sentence vectors for train/dev/test × {full, facts}, computed once        (all GPUs)
  L  lexical retriever: TF-IDF + one-vs-rest logistic regression, C chosen on dev, 5-fold cross-fitted on train (CPU)
  H  Hyena encoder: 5 cross-fitted models (asymmetric loss, label correlation, sentence dropout)   (folds spread over GPUs)
  S  shortlist: top-K of w·p_enc + (1−w)·p_lex, w chosen on out-of-fold TRAIN scores
  V  listwise Llama-3.2-3B QLoRA verifier trained on out-of-fold shortlists of the TRAIN split        (all GPUs, data parallel)
  I  verifier inference on dev and test, both views, two candidate orders                              (all GPUs)
  F  stacked fusion (logistic regression) + threshold, fitted on DEV only (threshold by 5-fold CV on dev)
  T  test evaluation: ablation table, 95% bootstrap CIs, v7-protocol subset, length buckets, top-20 sections;
     oracle ceiling diagnostics saved SEPARATELY (they use test labels and are not results)
Test labels are used only in stage T. Nothing is tuned on the test split.
"""
import os, sys, re, json, time, math, random, shutil, datetime, pickle, glob
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import numpy as np
import torch
import torch.nn.functional as F
import torch.distributed as dist
from hybrid_config import *
from legal_hybrid import *
from datasets import load_from_disk

# ───────────────────────── distributed setup ─────────────────────────
WORLD = int(os.environ.get("WORLD_SIZE", 1)); RANK = int(os.environ.get("RANK", 0)); LOCAL = int(os.environ.get("LOCAL_RANK", 0))
CUDA = torch.cuda.is_available()
if CUDA:
    torch.cuda.set_device(LOCAL)
DEV = torch.device(f"cuda:{LOCAL}" if CUDA else "cpu")
if WORLD > 1:   # long CPU stages (lexical model) run on one process while the others wait → generous timeout
    dist.init_process_group("nccl" if CUDA else "gloo",
                            timeout=datetime.timedelta(minutes=int(os.environ.get("HYBRID_COLLECTIVE_TIMEOUT_MIN", 120))))
MAIN = RANK == 0
T_START = time.time()


def log(*a):
    if MAIN:
        print(f"[{(time.time() - T_START) / 60:6.1f} min]", *a, flush=True)


_LAST_BEAT = [0.0]
def beat():
    now = time.time()
    if now - _LAST_BEAT[0] > 30:
        _LAST_BEAT[0] = now
        try: open(f"{WORK_DIR}/.heartbeat_{RANK}", "w").write(str(now))
        except OSError: pass


def barrier():
    if WORLD > 1:
        dist.barrier()


def gather(obj):
    if WORLD == 1:
        return [obj]
    out = [None] * WORLD; dist.all_gather_object(out, obj); return out


def bcast(obj):
    if WORLD == 1:
        return obj
    box = [obj]; dist.broadcast_object_list(box, src=0); return box[0]


def my_part(n):
    return list(range(RANK, n, WORLD))


def sync_grads(params, bucket=16_000_000):
    """Average gradients across GPUs (bucketed all-reduce); missing grads count as zero."""
    if WORLD == 1:
        return
    for p in params:
        if p.grad is None:
            p.grad = torch.zeros_like(p)
    group, size = [], 0
    def flush(g):
        flat = torch.cat([p.grad.reshape(-1).float() for p in g])
        dist.all_reduce(flat); flat.div_(WORLD)
        o = 0
        for p in g:
            n = p.grad.numel(); p.grad.copy_(flat[o:o + n].view_as(p.grad)); o += n
    for p in params:
        group.append(p); size += p.numel()
        if size >= bucket:
            flush(group); group, size = [], 0
    if group:
        flush(group)


random.seed(SEED + RANK); np.random.seed(SEED + RANK)
AMPERE = CUDA and torch.cuda.get_device_capability(DEV)[0] >= 8
DTYPE = (torch.bfloat16 if AMPERE else torch.float16) if CUDA else torch.float32
ACC = max(1, EFFECTIVE_BATCH // WORLD)
autocast = lambda: torch.autocast("cuda", dtype=DTYPE, enabled=CUDA)
log(f"GPUs in use: {WORLD} · {torch.cuda.get_device_name(DEV) if CUDA else 'cpu'} · dtype {DTYPE}")

# ───────────────────────── Hub sync ─────────────────────────
from huggingface_hub import HfApi, snapshot_download
REPO, api = "", None
if PUSH_TO_HUB and MAIN and not os.environ.get("HYBRID_OFFLINE"):
    try:
        api = HfApi(); REPO = HF_REPO or f"{api.whoami()['name']}/hyena-llama-legal-v8"
        api.create_repo(REPO, private=True, exist_ok=True)
        log(f"stage outputs + checkpoints → https://huggingface.co/{REPO} (private)")
    except Exception as e:
        api, REPO = None, ""
        log(f"[hub] disabled ({type(e).__name__}: {e}); outputs stay in {CKPT_DIR}")
REPO = bcast(REPO)


def push(local_dir, path_in_repo, msg):
    if api is None:
        return
    try:
        api.upload_folder(repo_id=REPO, folder_path=local_dir, path_in_repo=path_in_repo, commit_message=msg[:200])
        log(f"[hub] pushed {path_in_repo}")
    except Exception as e:
        log(f"[hub] push of {path_in_repo} failed ({type(e).__name__}); kept locally")


def stage_dir(name):
    return f"{CKPT_DIR}/{name}"


def stage_done(name):
    """True if the stage finished (locally, or restored from the Hub in a new session)."""
    d = stage_dir(name)
    if MAIN and not os.path.exists(f"{d}/done.json") and REPO:
        try:
            snapshot_download(REPO, allow_patterns=[f"checkpoints_v8/{name}/*"], local_dir=WORK_DIR)
            if os.path.exists(f"{d}/done.json"):
                log(f"[hub] restored stage {name}")
        except Exception:
            pass
    barrier()
    return os.path.exists(f"{d}/done.json")


def finish_stage(name, info, push_it=True):
    if MAIN:
        d = stage_dir(name); os.makedirs(d, exist_ok=True)
        json.dump({**info, "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")}, open(f"{d}/done.json", "w"), indent=1)
        log(f"stage {name} done · {json.dumps({k: v for k, v in info.items() if not isinstance(v, (list, dict)) or len(str(v)) < 300})}")
        if push_it:
            push(d, f"checkpoints_v8/{name}", f"stage {name}")
    barrier()

# ───────────────────────── data (identical to v6/v7) ─────────────────────────
meta = json.load(open(f"{DATA_DIR}/meta.json"))
LABEL_NAMES, STATUTE_DESC = meta["label_names"], meta["statute_desc"]
N_LABELS = len(LABEL_NAMES); L2I = {s: i for i, s in enumerate(LABEL_NAMES)}
SPLITS = {s: load_from_disk(f"{DATA_DIR}/{s}") for s in ["train", "dev", "test"]}
VIEWS = ["full", "facts"]
GOLD = {s: [set(LABEL_NAMES[j] for j in r["labels"]) for r in ds] for s, ds in SPLITS.items()}
log(f"data: " + " · ".join(f"{s} {len(ds)}" for s, ds in SPLITS.items()) + f" · labels {N_LABELS}")

if os.environ.get("HYBRID_TEST_FACTORY"):                 # CPU smoke tests only: tiny stand-in Llama + tokenizer
    import importlib
    _factory = importlib.import_module(os.environ["HYBRID_TEST_FACTORY"])
    tokenizer = _factory.tokenizer()
else:
    from transformers import AutoTokenizer
    if MAIN:
        snapshot_download(BASE_MODEL, allow_patterns=["*.json", "*.safetensors", "*.jinja", "tokenizer*"])
    barrier()
    MODEL_PATH = snapshot_download(BASE_MODEL, allow_patterns=["*.json", "*.safetensors", "*.jinja", "tokenizer*"])
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    pad = "<|finetune_right_pad_id|>"
    tokenizer.pad_token = pad if pad in tokenizer.get_vocab() else tokenizer.eos_token

_DOCS = {}
def make_doc(split, i, view):
    key = (split, i, view)
    if key not in _DOCS:
        sents, flags = view_sentences(SPLITS[split][i]["text"], view)
        _DOCS[key] = prepare_doc(sents, tokenizer, MAX_DOC_TOKENS, flags)
        if len(_DOCS) > 4000:
            _DOCS.pop(next(iter(_DOCS)))
    return _DOCS[key]

TITLES = {s: section_title(s, STATUTE_DESC) for s in LABEL_NAMES}
PRIOR = np.zeros(N_LABELS)
for r in SPLITS["train"]:
    PRIOR[list(r["labels"])] += 1
PRIOR /= max(1, len(SPLITS["train"]))
TOP20 = [LABEL_NAMES[j] for j in np.argsort(-PRIOR)[:20]]

# fold assignment shared by the lexical model and the encoder (cross-fitting)
FOLD = np.random.default_rng(SEED).permutation(len(SPLITS["train"])) % ENC_FOLDS

# ───────────────────────── stage E: InLegalBERT sentence vectors (computed once) ─────────────────────────
def load_embedder():
    if os.environ.get("HYBRID_TEST_EMBEDDER"):
        return SentenceEmbedder.load(os.environ["HYBRID_TEST_EMBEDDER"], DEV, window=60)
    pats = ["*.json", "*.txt", "*.safetensors", "pytorch_model.bin"]
    if MAIN:
        snapshot_download(LEGAL_ENCODER, allow_patterns=pats)
    barrier()
    return SentenceEmbedder.load(snapshot_download(LEGAL_ENCODER, allow_patterns=pats), DEV)

EMBEDDER = load_embedder()
os.makedirs(CACHE_DIR, exist_ok=True)
NEED_VECTORS = not stage_done("encoder")       # sentence vectors are only needed to train the encoders
if NEED_VECTORS and not os.path.exists(f"{CACHE_DIR}/done.json"):
    log("stage E: InLegalBERT sentence vectors for every judgment and view (computed once, reused by every stage)")
    for split, ds in SPLITS.items():
        for view in VIEWS:
            f = f"{CACHE_DIR}/{split}_{view}_r{RANK}of{WORLD}.pt"
            if os.path.exists(f):
                continue
            part = {}
            for n, i in enumerate(my_part(len(ds))):
                beat()
                part[i] = EMBEDDER(make_doc(split, i, view).sentences).to(torch.float16).cpu()
                if MAIN and (n + 1) % 500 == 0:
                    log(f"  E · {split}/{view} · {(n + 1) * WORLD}/{len(ds)}")
            torch.save(part, f + ".tmp"); os.replace(f + ".tmp", f)
    barrier()
    if MAIN:
        json.dump({"world": WORLD}, open(f"{CACHE_DIR}/done.json", "w"))
    barrier()
    log("stage E done")

_EMB = {}
def emb(split, view):
    if (split, view) not in _EMB:
        w = json.load(open(f"{CACHE_DIR}/done.json"))["world"]; d = {}
        for r in range(w):
            d.update(torch.load(f"{CACHE_DIR}/{split}_{view}_r{r}of{w}.pt"))
        _EMB[(split, view)] = [d[i] for i in range(len(SPLITS[split]))]
    return _EMB[(split, view)]

# ───────────────────────── stage L: lexical retriever (CPU, main process) ─────────────────────────
def lex_text(split, i, view):
    return " ".join(make_doc(split, i, view).sentences)

def new_lexical(C):
    from sklearn.pipeline import make_pipeline
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.multiclass import OneVsRestClassifier
    from sklearn.linear_model import LogisticRegression
    return make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9, sublinear_tf=True,
                                         max_features=LEX_MAX_FEATURES, strip_accents="unicode"),
                         OneVsRestClassifier(LogisticRegression(C=C, solver="liblinear", max_iter=1000), n_jobs=-1))

def indicator(split, idx):
    Y = np.zeros((len(idx), N_LABELS), dtype=int)
    for k, i in enumerate(idx):
        Y[k, list(SPLITS[split][i]["labels"])] = 1
    return Y

def probs_to_rows(P):
    return [{LABEL_NAMES[j]: float(p[j]) for j in range(N_LABELS)} for p in P]

if not stage_done("lexical"):
    if MAIN:
        import warnings; warnings.filterwarnings("ignore")
        ntr = len(SPLITS["train"]); tr_idx = list(range(ntr))
        Xtr = [lex_text("train", i, v) for v in VIEWS for i in tr_idx]; Ytr = np.vstack([indicator("train", tr_idx)] * 2)
        dev_txt = {v: [lex_text("dev", i, v) for i in range(len(SPLITS["dev"]))] for v in VIEWS}
        best = None
        for C in LEX_C_GRID:
            m = new_lexical(C).fit(Xtr, Ytr)
            rows = sum((probs_to_rows(m.predict_proba(dev_txt[v])) for v in VIEWS), [])
            g = GOLD["dev"] * 2; th = best_threshold(g, rows, LABEL_NAMES)
            f1 = micro_f1_raw(g, [decide(r, th) for r in rows])
            log(f"  L · C={C} · dev micro-F1 {f1:.3f} (threshold {th:.2f})")
            if best is None or f1 > best[0]:
                best = (f1, C, m, th)
        f1, C, final, th = best
        out = {}
        for split in ["dev", "test"]:
            for v in VIEWS:
                out[f"{split}_{v}"] = final.predict_proba([lex_text(split, i, v) for i in range(len(SPLITS[split]))]).astype(np.float32)
        oof = {v: np.zeros((ntr, N_LABELS), np.float32) for v in VIEWS}
        for f in range(ENC_FOLDS):
            beat()
            fit_idx = [i for i in tr_idx if FOLD[i] != f]; hold = [i for i in tr_idx if FOLD[i] == f]
            m = new_lexical(C).fit([lex_text("train", i, v) for v in VIEWS for i in fit_idx], np.vstack([indicator("train", fit_idx)] * 2))
            for v in VIEWS:
                oof[v][hold] = m.predict_proba([lex_text("train", i, v) for i in hold])
            log(f"  L · cross-fit fold {f + 1}/{ENC_FOLDS}")
        for v in VIEWS:
            out[f"train_{v}"] = oof[v]
        d = stage_dir("lexical"); os.makedirs(d, exist_ok=True)
        np.savez_compressed(f"{d}/p_lex.npz", **out)
        pickle.dump(final, open(f"{d}/lexical.pkl", "wb"))
    finish_stage("lexical", {"C": float(best[1]) if MAIN else None, "dev_micro_f1": round(best[0], 3) if MAIN else None,
                             "dev_threshold": best[3] if MAIN else None})
P_LEX = dict(np.load(f"{stage_dir('lexical')}/p_lex.npz"))
LEX_INFO = json.load(open(f"{stage_dir('lexical')}/done.json"))

# ───────────────────────── stage H: cross-fitted Hyena encoder ─────────────────────────
LABEL_DESC_EMB = EMBEDDER([f"{s}: {TITLES[s] or s}" for s in LABEL_NAMES]).float().cpu()
ENC_CONFIG = dict(in_dim=EMBEDDER.dim, d=ENC_DIM, n_layers=ENC_LAYERS, n_labels=N_LABELS, band_cutoff=BAND_CUTOFF, n_modes=N_MODES)

def multi_hot(labels):
    y = torch.zeros(N_LABELS, device=DEV); y[list(labels)] = 1.0; return y

def importance_loss(out, flags, labels):
    s = out["scores"].float(); S = s.numel(); parts = []
    if labels and S > 1:
        a = out["attn"][list(labels)].detach().float().mean(0).clamp_min(0).sqrt()
        parts.append(a / a.sum().clamp_min(1e-8))
    f = torch.tensor(flags, dtype=torch.float32, device=DEV)
    if S > 1 and 0 < f.sum() < S:
        parts.append(f / f.sum())
    if not parts:
        return s.sum() * 0.0
    t = (sum(parts) / len(parts)).clamp_min(1e-8)
    return (t * (t.log() - torch.log_softmax(s, 0))).sum()

def entropy_loss(out):
    a = out["attn"].float().clamp_min(1e-12); S = a.shape[-1]
    if S < 3:
        return a.sum() * 0.0
    return F.relu(ENTROPY_FLOOR - (-(a * a.log()).sum(-1) / math.log(S))).mean()

def new_encoder(seed, mean, std):
    torch.manual_seed(seed)
    e = LegalHyenaEncoder(**ENC_CONFIG, label_init=LABEL_DESC_EMB).to(DEV)
    e.emb_mean.copy_(mean.to(DEV)); e.emb_std.copy_(std.to(DEV)); return e

@torch.no_grad()
def enc_probs(e, split, idx, view):
    e.eval(); out = []
    for i in idx:
        beat()
        with autocast():
            out.append(torch.sigmoid(e(emb(split, view)[i].to(DEV))["logits"].float()).cpu().numpy())
    return np.stack(out) if out else np.zeros((0, N_LABELS))

def train_fold(f, mean, std):
    ntr = len(SPLITS["train"])
    fit_idx = [i for i in range(ntr) if FOLD[i] != f]; hold = [i for i in range(ntr) if FOLD[i] == f]
    e = new_encoder(SEED + 100 + f, mean, std)
    params = [p for p in e.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=ENC_LR, weight_decay=0.01)
    steps = ENC_EPOCHS * len(fit_idx) // EFFECTIVE_BATCH
    from transformers import get_cosine_schedule_with_warmup
    sched = get_cosine_schedule_with_warmup(opt, max(1, int(0.05 * steps)), max(1, steps))
    scaler = torch.amp.GradScaler("cuda", enabled=CUDA and not AMPERE)
    rng = random.Random(SEED + f); best = (-1.0, None, 0)
    hold_g = [GOLD["train"][i] for i in hold]
    for ep in range(ENC_EPOCHS):
        e.train(); order = fit_idx[:]; rng.shuffle(order); run = []
        for n, i in enumerate(order):
            beat()
            view = "facts" if rng.random() < FACTS_PROB else "full"
            x = emb("train", view)[i].to(DEV).float()
            flags = make_doc("train", i, view).flags
            if x.shape[0] > 4 and ENC_SENT_DROP > 0:          # sentence dropout (augmentation)
                keep = torch.rand(x.shape[0], generator=torch.Generator().manual_seed(rng.randrange(1 << 30))) > ENC_SENT_DROP
                keep[0] = True; x = x[keep.to(x.device)]; flags = [fl for fl, k in zip(flags, keep.tolist()) if k]
            labels = SPLITS["train"][i]["labels"]
            with autocast():
                out = e(x)
                loss = asl_loss(out["logits"], multi_hot(labels), ASL_GAMMA_NEG, ASL_CLIP) \
                    + 0.5 * importance_loss(out, flags, labels) + 0.5 * entropy_loss(out)
            scaler.scale(loss / EFFECTIVE_BATCH).backward(); run.append(loss.item())
            if (n + 1) % EFFECTIVE_BATCH == 0:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(); sched.step()
        P = enc_probs(e, "train", hold, "full")
        rows = probs_to_rows(P); th = best_threshold(hold_g, rows, LABEL_NAMES)
        f1 = micro_f1_raw(hold_g, [decide(r, th) for r in rows])
        print(f"[rank {RANK}] encoder fold {f + 1}/{ENC_FOLDS} · epoch {ep + 1}/{ENC_EPOCHS} · loss {np.mean(run):.4f} · "
              f"held-out micro-F1 {f1:.3f}", flush=True)
        if f1 > best[0]:
            best = (f1, {k: v.detach().cpu().clone() for k, v in e.state_dict().items()}, ep + 1)
    return best

if not stage_done("encoder"):
    d = stage_dir("encoder"); os.makedirs(d, exist_ok=True)
    with torch.no_grad():
        vecs = torch.cat([emb("train", "full")[i].float() for i in range(min(150, len(SPLITS["train"])))])
        mean, std = vecs.mean(0), vecs.std(0).clamp_min(1e-4)
    mean, std = bcast((mean, std))
    log(f"stage H: {ENC_FOLDS} cross-fitted Hyena encoders · {ENC_EPOCHS} epochs each · asymmetric loss · folds spread over {WORLD} GPU(s)")
    for f in range(ENC_FOLDS):
        if f % WORLD != RANK or os.path.exists(f"{d}/fold{f}.pt"):
            continue
        f1, sd, ep = train_fold(f, mean, std)
        torch.save({"state": sd, "held_out_f1": f1, "epoch": ep}, f"{d}/fold{f}.pt.tmp"); os.replace(f"{d}/fold{f}.pt.tmp", f"{d}/fold{f}.pt")
    barrier()
    folds = [torch.load(f"{d}/fold{f}.pt", map_location="cpu") for f in range(ENC_FOLDS)]
    encs = []
    for f in range(ENC_FOLDS):
        e = new_encoder(0, mean, std); e.load_state_dict(folds[f]["state"]); e.eval(); encs.append(e)
    # outputs for every judgment and view: probabilities, sentence importance, evidence sentence per section
    # TRAIN: the fold model that did NOT see the judgment (out-of-fold); DEV/TEST: mean of the 5 models
    local = {}
    for split in ["train", "dev", "test"]:
        for view in VIEWS:
            for i in my_part(len(SPLITS[split])):
                beat()
                x = emb(split, view)[i].to(DEV)
                with torch.no_grad(), autocast():
                    p, s, a = ensemble_forward([encs[FOLD[i]]] if split == "train" else encs, x)
                local[(split, view, i)] = (p.cpu().numpy().astype(np.float32), s.cpu().numpy().astype(np.float16),
                                           a.argmax(-1).cpu().numpy().astype(np.int32))
    if MAIN:
        allp = {}
        for part in gather(local):
            allp.update(part)
    else:
        gather(local)
    if MAIN:
        out = {}
        for split in ["train", "dev", "test"]:
            for view in VIEWS:
                n = len(SPLITS[split])
                out[f"{split}_{view}"] = {"p": np.stack([allp[(split, view, i)][0] for i in range(n)]),
                                          "scores": [allp[(split, view, i)][1] for i in range(n)],
                                          "evid": np.stack([allp[(split, view, i)][2] for i in range(n)])}
        pickle.dump(out, open(f"{d}/enc_outputs.pkl", "wb"))
        torch.save({"states": [fo["state"] for fo in folds], "config": ENC_CONFIG}, f"{d}/encoders.pt")
        for f in range(ENC_FOLDS):
            os.remove(f"{d}/fold{f}.pt")
        rows = sum((probs_to_rows(out[f"dev_{v}"]["p"]) for v in VIEWS), []); g = GOLD["dev"] * 2
        th = best_threshold(g, rows, LABEL_NAMES); f1 = micro_f1_raw(g, [decide(r, th) for r in rows])
        info = {"dev_micro_f1": round(f1, 3), "dev_threshold": th,
                "held_out_f1_per_fold": [round(fo["held_out_f1"], 3) for fo in folds], "best_epoch_per_fold": [fo["epoch"] for fo in folds]}
    else:
        info = {}
    gather(None)
    finish_stage("encoder", info)
_EMB.clear()                                   # free the sentence vectors (several GB of RAM) before loading Llama
ENC_OUT = pickle.load(open(f"{stage_dir('encoder')}/enc_outputs.pkl", "rb"))
ENC_INFO = json.load(open(f"{stage_dir('encoder')}/done.json"))
log(f"encoder ensemble · dev micro-F1 {ENC_INFO['dev_micro_f1']} · held-out per fold {ENC_INFO['held_out_f1_per_fold']}")

# ───────────────────────── stage S: shortlist ─────────────────────────
def recall_at_k(P, golds, k):
    hit = tot = 0
    for p, g in zip(P, golds):
        top = {LABEL_NAMES[j] for j in np.argsort(-p)[:k]}; hit += len(top & g); tot += len(g)
    return hit / max(1, tot)

if MAIN:
    g2 = GOLD["train"] * 2
    cand_w = np.round(np.arange(0, 1.01, 0.1), 1)
    rec = {float(w): recall_at_k(np.vstack([w * ENC_OUT[f"train_{v}"]["p"] + (1 - w) * P_LEX[f"train_{v}"] for v in VIEWS]), g2, SHORTLIST_K)
           for w in cand_w}
    BLEND_W = max(rec, key=rec.get)
else:
    BLEND_W = None
BLEND_W = bcast(BLEND_W)

_BL = {}
def blend(split, view):
    if (split, view) not in _BL:
        _BL[(split, view)] = BLEND_W * ENC_OUT[f"{split}_{view}"]["p"] + (1 - BLEND_W) * P_LEX[f"{split}_{view}"]
    return _BL[(split, view)]

if MAIN:
    gd = GOLD["dev"] * 2
    SL_REPORT = {}
    for name, P in [("lexical", np.vstack([P_LEX[f"dev_{v}"] for v in VIEWS])),
                    ("encoder", np.vstack([ENC_OUT[f"dev_{v}"]["p"] for v in VIEWS])),
                    ("blend", np.vstack([blend("dev", v) for v in VIEWS]))]:
        SL_REPORT[name] = {f"recall@{k}": round(recall_at_k(P, gd, k), 3) for k in (5, 10, SHORTLIST_K, 15)}
    log(f"stage S: blend weight w = {BLEND_W} (chosen on out-of-fold TRAIN recall@{SHORTLIST_K}) · dev shortlist recall: {SL_REPORT}")
else:
    SL_REPORT = None
SL_REPORT = bcast(SL_REPORT)

def candidates(split, view, i, inject=None):
    p = blend(split, view)[i]
    idx = shortlist(p, SHORTLIST_K)
    if inject is not None:                    # gold sections missing from the shortlist (training / oracle only)
        miss = [L2I[c] for c in inject if L2I[c] not in idx]
        if miss:
            neg = [j for j in idx if LABEL_NAMES[j] not in inject]
            for m in miss:
                if neg:
                    idx.remove(neg.pop())     # drop the lowest-ranked negative
                idx.append(m)
    return idx

def prompt_for(split, view, i, cand_idx, order=None):
    doc = make_doc(split, i, view); eo = ENC_OUT[f"{split}_{view}"]
    passages = key_passages(doc, eo["scores"][i].astype(np.float32), HEAD_TOKENS, PASSAGE_TOKENS)
    cands = [LABEL_NAMES[j] for j in (order if order is not None else cand_idx)]
    ev = {LABEL_NAMES[j]: doc.sentences[min(int(eo["evid"][i][j]), len(doc.sentences) - 1)][:EVIDENCE_CHARS] for j in cand_idx}
    return verify_user(passages, cands, TITLES, ev), cands

# ───────────────────────── stage V: listwise Llama verifier (QLoRA) ─────────────────────────
if os.environ.get("HYBRID_TEST_FACTORY"):
    llm = _factory.model(DEV)
else:
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    llm = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH, device_map={"": LOCAL}, dtype=DTYPE,
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                               bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE))
llm.config.pad_token_id = tokenizer.pad_token_id
llm.config.use_cache = False
for p in llm.parameters():
    p.requires_grad_(False)
llm.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict, get_peft_model_state_dict
torch.manual_seed(SEED)
llm = get_peft_model(llm, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                     target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
lora_params = [p for p in llm.parameters() if p.requires_grad]
VT = VerifierTokens(tokenizer)
log(f"Llama verifier loaded · LoRA {sum(p.numel() for p in lora_params) / 1e6:.1f} M params")

def save_lora(d):
    if MAIN:
        os.makedirs(d, exist_ok=True); torch.save(get_peft_model_state_dict(llm), f"{d}/lora.pt.tmp"); os.replace(f"{d}/lora.pt.tmp", f"{d}/lora.pt")

vd = stage_dir("verifier")
if not stage_done("verifier"):
    state = {}
    if MAIN and REPO and not os.path.exists(f"{stage_dir('verifier_progress')}/progress.json"):
        try: snapshot_download(REPO, allow_patterns=["checkpoints_v8/verifier_progress/*"], local_dir=WORK_DIR)
        except Exception: pass
    barrier()
    pd_ = stage_dir("verifier_progress")
    if os.path.exists(f"{pd_}/progress.json"):
        state = json.load(open(f"{pd_}/progress.json"))
        set_peft_model_state_dict(llm, torch.load(f"{pd_}/lora.pt", map_location=DEV))
        log(f"verifier: resuming after {state['examples_done']} examples")
    ntr = len(SPLITS["train"])
    plan = []
    for ep in range(VERIFIER_EPOCHS):
        perm = np.random.default_rng(SEED + 7 + ep).permutation(ntr).tolist()
        plan += perm
    plan = plan[: (len(plan) // WORLD) * WORLD]
    mine = plan[RANK::WORLD]
    n_updates = max(1, len(mine) // ACC)
    from transformers import get_cosine_schedule_with_warmup
    opt = torch.optim.AdamW(lora_params, lr=VERIFIER_LR, weight_decay=0.0)
    sched = get_cosine_schedule_with_warmup(opt, int(0.05 * n_updates), n_updates)
    skip = int(state.get("examples_done", 0)) // WORLD
    for _ in range(skip // ACC):
        sched.step()
    scaler = torch.amp.GradScaler("cuda", enabled=CUDA and not AMPERE)
    rng = random.Random(SEED + 11 + RANK); llm.train(); run = []; t0 = time.time(); updates = skip // ACC
    n_pos = n_all = 0
    log(f"stage V: {len(plan)} training judgments ({VERIFIER_EPOCHS} epoch) over {WORLD} GPU(s) · {n_updates} updates · "
        f"shortlist K={SHORTLIST_K} from OUT-OF-FOLD train scores" + (f" · resuming at {skip * WORLD}" if skip else ""))
    for step in range(skip, len(mine)):
        beat()
        i = mine[step]; view = "facts" if rng.random() < FACTS_PROB else "full"
        gold = GOLD["train"][i]
        idx = candidates("train", view, i, inject=gold if rng.random() < TRAIN_INJECT_GOLD else None)
        order = idx[:]; rng.shuffle(order)
        user, cands = prompt_for("train", view, i, idx, order)
        ids, pos, tgt = verifier_example(VT, SYSTEM, user, cands, [c in gold for c in cands])
        n_pos += int(tgt.sum()); n_all += len(tgt)
        with autocast():
            loss = verifier_loss(llm, VT, ids, pos, tgt, pos_weight=2.0)
        scaler.scale(loss / ACC).backward(); run.append(loss.item())
        _crash = os.environ.get("HYBRID_TEST_CRASH_AT")            # CPU tests only: simulated crash → restart from checkpoint
        if _crash and step == int(_crash) and not os.path.exists(f"{WORK_DIR}/.crashed_once"):
            barrier()
            if MAIN:
                open(f"{WORK_DIR}/.crashed_once", "w").close()
            barrier(); os._exit(1)
        if (step + 1) % ACC == 0:
            sync_grads(lora_params); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(lora_params, 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(); sched.step(); updates += 1
            if updates % CKPT_EVERY_UPDATES == 0 and step + 1 < len(mine):
                save_lora(pd_)
                if MAIN:
                    json.dump({"examples_done": (step + 1) * WORLD}, open(f"{pd_}/progress.json", "w"))
                    push(pd_, "checkpoints_v8/verifier_progress", "verifier progress")
                barrier()
        if (step + 1) % 50 == 0:
            done = step + 1 - skip; eta = (time.time() - t0) / done * (len(mine) - step - 1) / 60
            log(f"V · {(step + 1) * WORLD}/{len(plan)} · loss {np.mean(run[-50:]):.4f} · prompt {len(ids)} tokens · "
                f"positives {n_pos / max(1, n_all):.1%} of slots · ETA {eta:.0f} min")
    del opt, scaler, sched
    save_lora(vd)
    finish_stage("verifier", {"examples": len(plan), "updates": updates, "final_loss": round(float(np.mean(run[-100:] or [0])), 4)})
else:
    set_peft_model_state_dict(llm, torch.load(f"{vd}/lora.pt", map_location=DEV))
llm.eval()

# ───────────────────────── stage I: verifier inference on dev + test ─────────────────────────
def run_verifier(name, jobs):
    """jobs: list of (key, split, view, i, cand_idx). Saves {key: {section: P(yes)}} incrementally (resumable)."""
    d = stage_dir(name); os.makedirs(d, exist_ok=True)
    if MAIN and REPO and not glob.glob(f"{d}/part_r*of{WORLD}.json"):
        try: snapshot_download(REPO, allow_patterns=[f"checkpoints_v8/{name}/*"], local_dir=WORK_DIR)
        except Exception: pass
    barrier()
    f = f"{d}/part_r{RANK}of{WORLD}.json"
    done = json.load(open(f)) if os.path.exists(f) else {}
    mine = jobs[RANK::WORLD]; t0 = time.time(); new = 0
    for n, (key, split, view, i, idx) in enumerate(mine):
        if key in done:
            continue
        beat()
        ranked = sorted(idx, key=lambda j: -blend(split, view)[i][j])
        orders = [ranked, ranked[::-1]][:max(1, VERIFY_ORDERS)]
        q = {LABEL_NAMES[j]: 0.0 for j in idx}
        for order in orders:
            user, cands = prompt_for(split, view, i, idx, order)
            with autocast():
                for c, v in zip(cands, score_candidates(llm, VT, SYSTEM, user, cands)):
                    q[c] += v / len(orders)
        done[key] = q; new += 1
        if new % 25 == 0:
            json.dump(done, open(f + ".tmp", "w")); os.replace(f + ".tmp", f)
            log(f"I · {name} · {(n + 1) * WORLD}/{len(jobs)} · {(time.time() - t0) / max(1, new):.1f}s per judgment")
    json.dump(done, open(f + ".tmp", "w")); os.replace(f + ".tmp", f)
    barrier()
    if MAIN and new:
        push(d, f"checkpoints_v8/{name}", f"verifier outputs {name}")
    barrier()
    allq = {}
    for g in glob.glob(f"{d}/part_r*of{WORLD}.json"):
        allq.update(json.load(open(g)))
    return allq

N_TEST = min(N_EVAL, len(SPLITS["test"]))
jobs_dev = [(f"{v}:{i}", "dev", v, i, candidates("dev", v, i)) for i in range(len(SPLITS["dev"])) for v in VIEWS]
jobs_test = [(f"{v}:{i}", "test", v, i, candidates("test", v, i)) for i in range(N_TEST) for v in VIEWS]
log(f"stage I: verifier on {len(jobs_dev)} dev and {len(jobs_test)} test (judgment, view) pairs · {VERIFY_ORDERS} candidate orders")
Q_DEV = run_verifier("verifier_dev", jobs_dev)
Q_TEST = run_verifier("verifier_test", jobs_test)
Q_ORACLE = {}
if RUN_ORACLE_DIAGNOSTICS:     # DIAGNOSTIC ONLY: gold sections the shortlist missed are added (uses test labels)
    jobs_or = []
    for i in range(N_TEST):
        idx = candidates("test", "full", i, inject=GOLD["test"][i])
        if set(idx) != set(candidates("test", "full", i)):
            jobs_or.append((f"full:{i}", "test", "full", i, idx))
    Q_ORACLE = run_verifier("verifier_oracle", jobs_or)

# ───────────────────────── stages F + T (main process) ─────────────────────────
if MAIN:
    import pandas as pd
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import GroupKFold

    def feat_rows(split, view, i, q, idx):
        P_e, P_l, P_b = ENC_OUT[f"{split}_{view}"]["p"][i], P_LEX[f"{split}_{view}"][i], blend(split, view)[i]
        order = sorted(idx, key=lambda j: -P_b[j]); rank = {j: r / SHORTLIST_K for r, j in enumerate(order)}
        X = stack_features(P_e[idx], P_l[idx], P_b[idx], [q[LABEL_NAMES[j]] for j in idx], [rank[j] for j in idx], PRIOR[idx])
        return X, [LABEL_NAMES[j] for j in idx]

    def collect(split, Q, n, views, cand_fn=None):
        X, y, grp, meta_ = [], [], [], []
        for v in views:
            for i in range(n):
                key = f"{v}:{i}"
                if key not in Q:
                    continue
                idx = [L2I[c] for c in Q[key]]
                Xi, cands = feat_rows(split, v, i, Q[key], idx)
                X.append(Xi); y += [c in GOLD[split][i] for c in cands]; grp += [i] * len(cands); meta_.append((v, i, cands))
        return np.vstack(X), np.array(y, int), np.array(grp), meta_

    def rows_from(meta_, probs):
        rows, o = {}, 0
        for v, i, cands in meta_:
            rows[(v, i)] = dict(zip(cands, probs[o:o + len(cands)])); o += len(cands)
        return rows

    # ── F: stacker fitted on DEV; threshold chosen on 5-fold cross-validated dev predictions ──
    Xd, yd, gd_, md = collect("dev", Q_DEV, len(SPLITS["dev"]), VIEWS)
    oof = np.zeros(len(yd))
    for tr, va in GroupKFold(5).split(Xd, yd, gd_):
        oof[va] = LogisticRegression(C=1.0, max_iter=5000).fit(Xd[tr], yd[tr]).predict_proba(Xd[va])[:, 1]
    dev_rows = rows_from(md, oof); keys = list(dev_rows)
    dev_g = [GOLD["dev"][i] for _, i in keys]
    TH = best_threshold(dev_g, [dev_rows[k] for k in keys], LABEL_NAMES)
    f_plain = micro_f1_raw(dev_g, [decide(dev_rows[k], TH) for k in keys])
    f_one = micro_f1_raw(dev_g, [decide(dev_rows[k], TH, True) for k in keys])
    AT_LEAST_ONE = f_one > f_plain
    STACKER = LogisticRegression(C=1.0, max_iter=5000).fit(Xd, yd)
    log(f"stage F: stacker fitted on {len(yd)} dev (judgment, candidate) rows · threshold {TH:.2f} · at-least-one rule "
        f"{'on' if AT_LEAST_ONE else 'off'} · cross-validated dev micro-F1 {max(f_plain, f_one):.3f}")

    # dev thresholds for every baseline row (nothing is tuned on test)
    def all_label_rows(P):
        return [{LABEL_NAMES[j]: float(p[j]) for j in range(N_LABELS)} for p in P]
    dev_P = {"lexical": np.vstack([P_LEX[f"dev_{v}"] for v in VIEWS]), "encoder": np.vstack([ENC_OUT[f"dev_{v}"]["p"] for v in VIEWS]),
             "blend": np.vstack([blend("dev", v) for v in VIEWS])}
    g2 = GOLD["dev"] * 2
    TH_B = {k: best_threshold(g2, all_label_rows(P), LABEL_NAMES) for k, P in dev_P.items()}
    q_rows_dev = [Q_DEV[f"{v}:{i}"] for v in VIEWS for i in range(len(SPLITS["dev"]))]
    TH_B["verifier"] = best_threshold(g2, q_rows_dev, LABEL_NAMES)
    ALPHA_V7 = 0.6
    def v7_rows(split, Q, v, i):
        P_e = ENC_OUT[f"{split}_{v}"]["p"][i]
        return {c: ALPHA_V7 * float(P_e[L2I[c]]) + (1 - ALPHA_V7) * q for c, q in Q[f"{v}:{i}"].items()}
    TH_B["v7fusion"] = best_threshold(g2, [v7_rows("dev", Q_DEV, v, i) for v in VIEWS for i in range(len(SPLITS["dev"]))], LABEL_NAMES)
    log(f"dev thresholds: {TH_B}")

    # ── T: test evaluation ──
    Xt, yt, gt_, mt = collect("test", Q_TEST, N_TEST, VIEWS)
    test_rows = rows_from(mt, STACKER.predict_proba(Xt)[:, 1])
    SYSTEMS = ["TF-IDF lexical model alone", "Hyena encoder ensemble alone", "Lexical + encoder blend (no LLM)",
               "Llama verifier alone (on shortlist)", "Fixed-weight fusion α=0.6 (v7-style)", "v8: stacked fusion (ours)"]

    def predictions(view, i):
        P = {"lexical": P_LEX[f"test_{view}"][i], "encoder": ENC_OUT[f"test_{view}"]["p"][i], "blend": blend("test", view)[i]}
        pr = {SYSTEMS[0]: decide(all_label_rows([P["lexical"]])[0], TH_B["lexical"]),
              SYSTEMS[1]: decide(all_label_rows([P["encoder"]])[0], TH_B["encoder"]),
              SYSTEMS[2]: decide(all_label_rows([P["blend"]])[0], TH_B["blend"]),
              SYSTEMS[3]: decide(Q_TEST[f"{view}:{i}"], TH_B["verifier"]),
              SYSTEMS[4]: decide(v7_rows("test", Q_TEST, view, i), TH_B["v7fusion"]),
              SYSTEMS[5]: decide(test_rows[(view, i)], TH, AT_LEAST_ONE)}
        fused = test_rows[(view, i)]
        ranked = sorted(fused, key=lambda c: -fused[c]) + [LABEL_NAMES[j] for j in np.argsort(-P["blend"]) if LABEL_NAMES[j] not in fused]
        return pr, ranked

    recs = []
    for view in VIEWS:
        for i in range(N_TEST):
            pr, ranked = predictions(view, i)
            recs.append({"view": view, "i": i, "id": SPLITS["test"][i]["id"], "gold": sorted(GOLD["test"][i], key=sec_key),
                         "pred": {k: sorted(v, key=sec_key) for k, v in pr.items()}, "ranked": ranked[:10],
                         "fused": {c: round(float(p), 4) for c, p in test_rows[(view, i)].items()},
                         "verifier": {c: round(float(p), 4) for c, p in Q_TEST[f"{view}:{i}"].items()},
                         "tokens": len(make_doc("test", i, "full").ids)})

    def table(rs, tag):
        out = []
        for view in VIEWS:
            vr = [r for r in rs if r["view"] == view]; G = [set(r["gold"]) for r in vr]
            for s in SYSTEMS:
                Pp = [set(r["pred"][s]) for r in vr]
                out.append({"subset": tag, "view": view, "system": s, "judgments": len(vr), **micro_prf(G, Pp),
                            "macro-F1": macro_f1(G, Pp, LABEL_NAMES), "doc-F1": round(float(np.mean([doc_f1(a, b) for a, b in zip(G, Pp)])), 3)})
        return out

    results = pd.DataFrame(table(recs, f"all {N_TEST} test judgments") +
                           table([r for r in recs if r["i"] < V7_SUBSET], f"first {min(V7_SUBSET, N_TEST)} (v7 protocol)"))
    ours = SYSTEMS[5]
    extra = []
    for view in VIEWS:
        vr = [r for r in recs if r["view"] == view]; G = [set(r["gold"]) for r in vr]
        Po = [set(r["pred"][ours]) for r in vr]
        ci = bootstrap_ci(G, Po, n=BOOTSTRAP)
        d_enc = bootstrap_ci(G, Po, [set(r["pred"][SYSTEMS[1]]) for r in vr], n=BOOTSTRAP)
        d_v7 = bootstrap_ci(G, Po, [set(r["pred"][SYSTEMS[4]]) for r in vr], n=BOOTSTRAP)
        top = set(TOP20)
        rk = ranking_metrics(G, [r["ranked"] for r in vr])
        sl = recall_at_k(blend("test", view)[:N_TEST], [GOLD["test"][i] for i in range(N_TEST)], SHORTLIST_K)
        extra.append({"view": view, "micro-F1": micro_prf(G, Po)["f1"], "95% CI": ci,
                      "Δ vs encoder alone (95% CI)": d_enc, "Δ vs v7-style fusion (95% CI)": d_v7, **rk,
                      f"shortlist recall@{SHORTLIST_K}": round(sl, 3),
                      "top-20 micro-F1": micro_prf([g & top for g in G], [p & top for p in Po])["f1"],
                      "top-20 macro-F1": macro_f1([g & top for g in G], [p & top for p in Po], TOP20)})
    extra = pd.DataFrame(extra)

    by_len = []
    for view in VIEWS:
        vr = sorted([r for r in recs if r["view"] == view], key=lambda r: r["tokens"])
        n = len(vr)
        for name, part in zip(["short", "medium", "long"], [vr[: n // 3], vr[n // 3: 2 * n // 3], vr[2 * n // 3:]]):
            if part:
                by_len.append({"view": view, "length": f"{name} (median ~{int(np.median([r['tokens'] for r in part])):,} tokens)",
                               "judgments": len(part), "micro-F1": micro_prf([set(r["gold"]) for r in part], [set(r["pred"][ours]) for r in part])["f1"]})
    by_len = pd.DataFrame(by_len)

    # ── oracle diagnostics: use TEST labels → saved separately, never part of the results ──
    diag = []
    if RUN_ORACLE_DIAGNOSTICS:
        vr = [r for r in recs if r["view"] == "full"]; G = [set(r["gold"]) for r in vr]
        sl = recall_at_k(blend("test", "full")[:N_TEST], G, SHORTLIST_K)
        diag.append({"diagnostic": f"perfect verifier on the real shortlist (precision 1, recall = shortlist recall {sl:.3f})",
                     "micro-F1": round(2 * sl / (1 + sl), 3)})
        rows_t = [test_rows[("full", i)] for i in range(N_TEST)]
        th_o = best_threshold(G, rows_t, LABEL_NAMES)
        diag.append({"diagnostic": f"threshold chosen on TEST ({th_o:.2f}) instead of dev ({TH:.2f})",
                     "micro-F1": round(micro_f1_raw(G, [decide(r, th_o, AT_LEAST_ONE) for r in rows_t]), 3)})
        Qo = {**{f"full:{i}": Q_TEST[f"full:{i}"] for i in range(N_TEST)}, **Q_ORACLE}
        Xo, _, _, mo = collect("test", Qo, N_TEST, ["full"])
        ro = rows_from(mo, STACKER.predict_proba(Xo)[:, 1])
        diag.append({"diagnostic": "gold sections the shortlist missed added to the shortlist (oracle retrieval), same stacker + dev threshold",
                     "micro-F1": round(micro_f1_raw(G, [decide(ro[("full", i)], TH, AT_LEAST_ONE) for i in range(N_TEST)]), 3)})
        diag.append({"diagnostic": "v8 as reported (for reference)", "micro-F1": micro_prf(G, [set(r["pred"][ours]) for r in vr])["f1"]})
    diag = pd.DataFrame(diag)

    # ── export ──
    os.makedirs(OUT_DIR, exist_ok=True)
    results.to_csv(f"{OUT_DIR}/ablation_results.csv", index=False); extra.to_csv(f"{OUT_DIR}/v8_summary_metrics.csv", index=False)
    by_len.to_csv(f"{OUT_DIR}/results_by_length.csv", index=False)
    if len(diag):
        diag.to_csv(f"{OUT_DIR}/DIAGNOSTICS_ORACLE_not_results.csv", index=False)
    json.dump(recs, open(f"{OUT_DIR}/eval_predictions.json", "w"))
    enc_states = torch.load(f"{stage_dir('encoder')}/encoders.pt", map_location="cpu")
    torch.save({"encoder_states": enc_states["states"], "encoder_config": enc_states["config"], "legal_encoder": LEGAL_ENCODER,
                "label_names": LABEL_NAMES, "statute_desc": STATUTE_DESC, "system_prompt": SYSTEM, "base_model": BASE_MODEL,
                "max_doc_tokens": MAX_DOC_TOKENS, "shortlist_k": SHORTLIST_K, "head_tokens": HEAD_TOKENS,
                "passage_tokens": PASSAGE_TOKENS, "evidence_chars": EVIDENCE_CHARS, "verify_orders": VERIFY_ORDERS,
                "version": 8}, f"{OUT_DIR}/hybrid_model_v8.pt")
    pickle.dump({"lexical": pickle.load(open(f"{stage_dir('lexical')}/lexical.pkl", "rb")), "stacker": STACKER, "threshold": TH,
                 "at_least_one": AT_LEAST_ONE, "blend_w": BLEND_W, "label_prior": PRIOR}, open(f"{OUT_DIR}/lexical_and_stacker.pkl", "wb"))
    llm.save_pretrained(f"{OUT_DIR}/llama_lora"); tokenizer.save_pretrained(f"{OUT_DIR}/llama_lora")
    for f in ("legal_hybrid.py", "hybrid_config.py"):
        shutil.copy(os.path.join(os.path.dirname(os.path.abspath(__file__)), f), OUT_DIR)
    json.dump({"lexical": LEX_INFO, "encoder": ENC_INFO, "blend_w": BLEND_W, "shortlist_dev": SL_REPORT, "threshold": TH,
               "at_least_one": AT_LEAST_ONE, "baseline_thresholds": TH_B, "n_test": N_TEST}, open(f"{OUT_DIR}/training_summary.json", "w"), indent=1)
    pd.set_option("display.width", 250); pd.set_option("display.max_columns", 20)
    log("\n" + results.to_string(index=False) + "\n\n" + extra.to_string(index=False) + "\n\nby length:\n" + by_len.to_string(index=False)
        + ("\n\nDIAGNOSTICS ONLY (use test labels, not results):\n" + diag.to_string(index=False) if len(diag) else ""))
    push(OUT_DIR, "final", "final model + results")
barrier()
log("ALL DONE")
if WORLD > 1:
    dist.destroy_process_group()

In [ ]:
%%writefile /kaggle/working/sync_checkpoints.py
"""
sync_checkpoints.py: run this on YOUR LAPTOP while Kaggle trains.
It watches your private Hugging Face repo and downloads every new checkpoint automatically
(only changed files are transferred). Stops by itself when the final evaluation results arrive.

Setup (once):   pip install -U huggingface_hub
Run:            python sync_checkpoints.py --repo <your-hf-username>/hyena-llama-legal --token hf_xxx
                (or set the HF_TOKEN environment variable, or run `huggingface-cli login` once)
Options:        --out <folder> (default: ./<repo-name>)   --every 300   --once
"""
import argparse, json, os, time, glob
from huggingface_hub import HfApi, snapshot_download

ap = argparse.ArgumentParser()
ap.add_argument("--repo", required=True, help="e.g. yourname/hyena-llama-legal")
ap.add_argument("--token", default=os.environ.get("HF_TOKEN"), help="Hugging Face token with READ access")
ap.add_argument("--out", default=None, help="local folder (default: ./<repo-name>, so each version stays separate)")
ap.add_argument("--every", type=int, default=300, help="seconds between checks")
ap.add_argument("--once", action="store_true", help="sync once and exit")
a = ap.parse_args()
a.out = a.out or os.path.join(".", a.repo.split("/")[-1])

api, last = HfApi(token=a.token), None
os.makedirs(a.out, exist_ok=True)
print(f"watching https://huggingface.co/{a.repo} → {os.path.abspath(a.out)} (Ctrl+C to stop)")
while True:
    stamp = time.strftime("%H:%M:%S")
    try:
        sha = api.repo_info(a.repo, token=a.token).sha
        if sha != last:
            snapshot_download(a.repo, local_dir=a.out, token=a.token)
            last = sha
            print(f"[{stamp}] synced commit {str(sha)[:8]}")
            for st in sorted(glob.glob(f"{a.out}/checkpoints_v8/*/done.json")):
                s = json.load(open(st)); name = os.path.basename(os.path.dirname(st))
                print(f"    stage {name:10s} finished · saved {s.get('saved_at')}")
            pr = f"{a.out}/checkpoints_v8/verifier_progress/progress.json"
            if os.path.exists(pr):
                print(f"    verifier training: {json.load(open(pr))['examples_done']} judgments done")
            res = f"{a.out}/final/ablation_results.csv"
            if os.path.exists(res):
                print("\nFinal results arrived:\n" + open(res).read())
                print(f"Trained model: {os.path.abspath(a.out)}/final  (hybrid_model_v8.pt + lexical_and_stacker.pkl + llama_lora/)")
                break
        else:
            print(f"[{stamp}] no new checkpoint")
    except KeyboardInterrupt:
        break
    except Exception as e:
        print(f"[{stamp}] could not reach the Hub ({type(e).__name__}); retrying later")
    if a.once:
        break
    time.sleep(a.every)

## 5 · Prepare data (restored from your data repo: the same v4 data as v6/v7)

In [ ]:
!cd /kaggle/working && PYTHONUNBUFFERED=1 python prepare_data.py

## 6 · Train + evaluate on all GPUs
Training runs **in the background** and writes to `train_log.txt`; the cell below shows the progress lines once a minute. Stopping the cell does **not** stop training; run it again to keep watching. If a process dies or stalls for 40 min, training restarts from the last finished stage (up to 5 times).

In [ ]:
%%writefile /kaggle/working/run_training.py
# Background trainer supervisor: runs torchrun on all GPUs, writes everything to train_log.txt (never to the notebook),
# restarts from the last checkpoint if a process dies or stops making progress. Started by the notebook; survives
# browser disconnects and notebook-output stalls.
import os, sys, time, glob, signal, subprocess, json
W = os.environ.get("HYBRID_WORK_DIR", "/kaggle/working")
LOG, STATUS = f"{W}/train_log.txt", f"{W}/.trainer_status.json"
MAX_ATTEMPTS = int(os.environ.get("HYBRID_MAX_ATTEMPTS", 5))
STALL_MIN = float(os.environ.get("HYBRID_STALL_MIN", 40))        # no progress for this long → restart

def status(**kw):
    json.dump({"pid": os.getpid(), "time": time.time(), **kw}, open(STATUS + ".tmp", "w")); os.replace(STATUS + ".tmp", STATUS)

def ngpu():
    try:
        out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
        return max(1, sum(l.startswith("GPU") for l in out.splitlines()))
    except FileNotFoundError:
        return int(os.environ.get("HYBRID_NPROC", 1))

def last_progress(t0):
    files = glob.glob(f"{W}/.heartbeat_*") + [LOG]
    return max([t0] + [os.path.getmtime(f) for f in files if os.path.exists(f)])

def kill(p=None):
    """Kill torchrun and every training worker (workers run in their own process groups, so search /proc)."""
    if p is not None:
        try: os.killpg(p.pid, signal.SIGKILL)
        except ProcessLookupError: pass
    for d in os.listdir("/proc"):
        if not d.isdigit() or int(d) == os.getpid():
            continue
        try:
            argv = open(f"/proc/{d}/cmdline", "rb").read().split(b"\0")
        except OSError:
            continue
        if any(x == b"train_v8.py" or x.endswith(b"/train_v8.py") for x in argv[1:]):   # exact argument, so shells that merely mention it are spared
            try: os.kill(int(d), signal.SIGKILL)
            except ProcessLookupError: pass
    time.sleep(5)

import fcntl
_lock = open(f"{W}/.trainer.lock", "w")
try:
    fcntl.flock(_lock, fcntl.LOCK_EX | fcntl.LOCK_NB)            # only one supervisor at a time
except BlockingIOError:
    print("another trainer is already running"); sys.exit(0)
status(state="starting")
N = ngpu()
env = dict(os.environ, NCCL_P2P_DISABLE="1", NCCL_IB_DISABLE="1", PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false",
           HF_HUB_DISABLE_PROGRESS_BARS="1", TORCH_NCCL_ASYNC_ERROR_HANDLING="1")
kill()                                                            # leftovers from an earlier launch would hold GPU memory
for attempt in range(1, MAX_ATTEMPTS + 1):
    for f in glob.glob(f"{W}/.heartbeat_*"):
        os.remove(f)
    cmd = ["torchrun", "--nnodes=1", f"--nproc_per_node={N}", "--node-rank=0", "--rdzv-backend=static",
           "--master-addr=127.0.0.1", f"--master-port={29500 + attempt}", "train_v8.py"]
    with open(LOG, "a") as lf:
        lf.write(f"\n===== attempt {attempt}/{MAX_ATTEMPTS} on {N} GPU(s) · {time.strftime('%H:%M:%S')} =====\n"); lf.flush()
        p = subprocess.Popen(cmd, cwd=W, env=env, stdout=lf, stderr=subprocess.STDOUT, start_new_session=True)
    status(state="running", attempt=attempt, gpus=N)
    t0, stalled = time.time(), False
    while p.poll() is None:
        time.sleep(30)
        if time.time() - last_progress(t0) > STALL_MIN * 60:
            stalled = True; kill(p); p.wait(); break
    rc = p.returncode
    if rc == 0 and not stalled:
        status(state="done", attempt=attempt)
        open(LOG, "a").write("\n✓ training + evaluation finished\n"); sys.exit(0)
    kill(p)                                                       # free any GPU still held by a worker
    why = f"no progress for {STALL_MIN:.0f} min" if stalled else f"exit code {rc}"
    open(LOG, "a").write(f"\n⚠ run stopped ({why}). Restarting from the last checkpoint in 20 s …\n")
    status(state="restarting", attempt=attempt, reason=why); time.sleep(20)
status(state="failed", attempt=MAX_ATTEMPTS)
open(LOG, "a").write(f"\n✗ training failed {MAX_ATTEMPTS} times\n"); sys.exit(1)

In [ ]:
import subprocess, os, time, json, re
W = "/kaggle/working"; LOG, STATUS = f"{W}/train_log.txt", f"{W}/.trainer_status.json"

def trainer_alive():
    try:
        pid = json.load(open(STATUS))["pid"]; os.kill(pid, 0)
        return json.load(open(STATUS))["state"] in ("starting", "running", "restarting")
    except Exception:
        return False

if trainer_alive():
    print("trainer already running → watching it")
else:
    subprocess.Popen(["python", "run_training.py"], cwd=W, stdout=subprocess.DEVNULL, stderr=open(f"{W}/supervisor_err.txt", "a"),
                     start_new_session=True)
    print("trainer started in the background"); time.sleep(5)

KEEP = re.compile(r"^(\[\s*\d|\[rank|=====|⚠|✓|✗|WARNING|\S*Error|torch.*Error|\s+(TF-IDF|Hyena|Lexical|Llama|Fixed|v8|subset|view|full|facts))")
pos = 0
while True:
    if os.path.exists(LOG):
        with open(LOG, errors="replace") as f:
            f.seek(pos); new = f.read(); pos = f.tell()
        shown = [l for l in new.splitlines() if KEEP.match(l)]
        if shown:
            print("\n".join(shown[-60:]), flush=True)
    st = json.load(open(STATUS))["state"] if os.path.exists(STATUS) else "starting"
    if not trainer_alive():
        if st not in ("done", "failed"):
            st = "failed"; print(open(f"{W}/supervisor_err.txt", errors="replace").read()[-2000:])
        print(f"\ntrainer finished: {st}")
        if st == "failed":
            print(open(LOG, errors="replace").read()[-4000:])
        break
    time.sleep(60)

## 7 · Results

In [ ]:
import sys, os, json, pandas as pd, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, "/kaggle/working")
from hybrid_config import *
pd.set_option("display.max_colwidth", 70); pd.set_option("display.width", 250)
summary = json.load(open(f"{OUT_DIR}/training_summary.json"))
print("Lexical model (dev):", summary["lexical"]); print("Encoder ensemble (dev):", summary["encoder"])
print(f"Blend weight w = {summary['blend_w']} · shortlist recall on dev:", summary["shortlist_dev"])
print(f"Stacker threshold {summary['threshold']:.2f} (chosen on dev) · at-least-one rule {summary['at_least_one']}")
res = pd.read_csv(f"{OUT_DIR}/ablation_results.csv")
for subset in res["subset"].unique():
    for view in ["full", "facts"]:
        print(f"\n{subset} · {'FULL TEXT (citation numbers hidden)' if view == 'full' else 'FACTS-ONLY (citing sentences removed)'}")
        display(res[(res.subset == subset) & (res.view == view)].drop(columns=["subset", "view"]))
print("\nv8 · confidence intervals, ranking and frequent-section metrics:"); display(pd.read_csv(f"{OUT_DIR}/v8_summary_metrics.csv"))
print("v8 · micro-F1 by judgment length:"); display(pd.read_csv(f"{OUT_DIR}/results_by_length.csv"))
diag = f"{OUT_DIR}/DIAGNOSTICS_ORACLE_not_results.csv"
if os.path.exists(diag):
    print("\n" + "!" * 100 + "\nORACLE DIAGNOSTICS: these use TEST labels to measure headroom for design decisions."
          "\nThey are NOT results and must not be reported as the model's performance.\n" + "!" * 100)
    display(pd.read_csv(diag))

main = res[res.subset == res.subset.iloc[0]]
fig, ax = plt.subplots(figsize=(10, 3.8))
systems = list(dict.fromkeys(main.system)); x = np.arange(len(systems))
for k, (view, col) in enumerate([("full", "#9A9A9A"), ("facts", "#2E5C8A")]):
    v = main[main.view == view].set_index("system").loc[systems, "f1"].values
    b = ax.bar(x + (k - 0.5) * 0.38, v, 0.38, label=view, color=col)
    ax.bar_label(b, fmt="%.3f", fontsize=7)
ax.set_xticks(x); ax.set_xticklabels([s.replace(" (", "\n(") for s in systems], fontsize=7)
ax.set_ylabel("test micro-F1"); ax.set_title(f"v8 ablation · {main.subset.iloc[0]}"); ax.legend(); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/v8_ablation.png", dpi=150); plt.show()

## 8 · Load the trained model (exactly as on a laptop) · demo

In [ ]:
import torch, re, glob, time
from legal_hybrid import *
from datasets import load_from_disk
tok, llm, encoders, ck = load_trained(OUT_DIR, "cuda:0")
predict_case = Predictor(tok, llm, encoders, ck)
print(f"loaded v8 · VRAM {torch.cuda.memory_allocated()/1024**3:.2f} GB (Llama 4-bit + LoRA + InLegalBERT + 5 Hyena encoders)")

def read_document(path):
    if path.lower().endswith(".pdf"):
        import pymupdf; return "\n".join(p.get_text() for p in pymupdf.open(path))
    return open(path, encoding="utf-8", errors="ignore").read()

def analyse_case(text, hide_cited=True, view="full"):
    torch.cuda.reset_peak_memory_stats(); t0 = time.time()
    r = predict_case(text, hide_cited=hide_cited, view=view)
    print("=" * 90); print(f"{r['doc_tokens']:,} tokens · {r['sentences']} sentences")
    print("\nSUGGESTED IPC SECTIONS")
    for s in r["sections"]:
        print(f"  • {s} ({r['titles'].get(s, '')}) · fused {r['fused'][s]:.2f} · verifier {r['verifier'][s]:.2f}")
        print(f"      evidence: {r['evidence'][s][:180]}")
    print("\nOther shortlisted sections:", ", ".join(f"{c} {p:.2f}" for c, p in r["fused"].items() if c not in r["sections"]))
    print(f"\ntime {time.time() - t0:.1f}s · peak VRAM {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")
    print("Decision support only: a qualified lawyer must verify every section.")
    return r

test = load_from_disk(f"{DATA_DIR}/test")
row = test[0]
print("GOLD:", sorted([ck["label_names"][i] for i in row["labels"]], key=sec_key))
_ = analyse_case(" ".join(row["text"]), hide_cited=False)
for p in sorted(glob.glob("/kaggle/input/**/*.pdf", recursive=True))[:3]:
    print(f"\n##### {p}"); _ = analyse_case(read_document(p), hide_cited=True)

## 9 · Push results + package everything

In [ ]:
from huggingface_hub import HfApi
!cd /kaggle/working && cp train_log.txt hybrid_legal_v8/ 2>/dev/null; zip -qr hybrid_legal_v8.zip hybrid_legal_v8 && ls -lh hybrid_legal_v8.zip
try:
    api = HfApi(); repo = HF_REPO or f"{api.whoami()['name']}/hyena-llama-legal-v8"
    api.upload_folder(repo_id=repo, folder_path=OUT_DIR, path_in_repo="final", commit_message="results + plots")
    print(f"pushed to https://huggingface.co/{repo}")
except Exception as e:
    print(f"Hub push skipped ({type(e).__name__}). Download hybrid_legal_v8.zip from the Output tab instead.")

### Notes
* **Laptop / server use:** `from legal_hybrid import load_trained, Predictor; predict = Predictor(*load_trained("hybrid_legal_v8", "cuda:0"))`,
  then `predict(text)` returns the sections, fused and verifier probabilities, titles, evidence sentences and key passages.
* **What is tuned where:** lexical C, blend weight, encoder epochs → train/dev; stacker + threshold → dev (cross-validated);
  nothing on test. The oracle diagnostics are the only place test labels are used before scoring, and they are kept apart.
* **Restart / resume:** every stage writes `checkpoints_v8/<stage>/done.json` (and pushes it); delete a stage folder in the
  Hub repo to recompute it. Delete `cache_v8/` to recompute the sentence vectors.
* **Faster first pass:** `ENC_EPOCHS = 4`, `VERIFY_ORDERS = 1`, `N_EVAL = 150`, `RUN_ORACLE_DIAGNOSTICS = False`.
* **CUDA out of memory in verifier training:** lower `PASSAGE_TOKENS` to 1024 or `SHORTLIST_K` to 10.